# CLAM-TR LoRA v1: UNI v1 Fine-Tuning with LoRA on PANDA**Goal**: Fine-tune UNI v1 (ViT-L/16, 300M params) with LoRA adapters on 9,128 PANDA slides.Determine whether task-specific encoder adaptation improves over frozen features.**Stages**:- Stage 0: Attention-based patch selection (Top-128 per slide)- Stage 1: Patch pre-extraction from .tiff files- Stage 2: LoRA training (5-fold CV)- Stage 3: Full feature re-extraction with LoRA adapters- Stage 4: V20 pipeline re-run with LoRA features

## ⚡ A100 80GB + 167GB RAM — EVERYTHING MAXED, K=128

### Resources Used
```
GPU:  ████████░░░░░░░░░░░░  23/80 GB  (ViT-L + 128 patches — MIL max)
RAM:  ██████████████████████ 155/167 GB (ALL 9128 slides cached in RAM)
SSD:  ██░░░░░░░░░░░░░░░░░░  19/235 GB  (robust copy from Drive)
```

### Data Flow (zero I/O during training)
```
Drive ──→ SSD (robust: waves of 200, 5 retries, atomic writes)
  SSD ──→ RAM (~150GB global cache, loaded ONCE for ALL 5 folds)
    RAM ──→ GPU (CUDA prefetcher, async stream, TF32)
```

### Why GPU shows 23/80
MIL = 1 slide at a time. ViT-L/16 with 128 patches maxes at ~23GB.
The A100's power is its 312 TFLOPS + 2TB/s bandwidth, not VRAM size.
Same K=128 that takes 3.5h/epoch on L4 takes ~45min on A100.

### Checkpoints: fully compatible with L4 K=128 runs


## 🔧 OPTIMIZED — A100 80GB Colab Edition (377CU friendly)

This is the OPTIMIZED variant of `CLAM_TR_LoRA_v1.ipynb`. Changes vs original:

**Critical fix:**
- 🔐 HF token now read from `google.colab.userdata` (was hardcoded — rotate your old token!).

**Performance fixes:**
- 💾 LoRA epoch checkpoints save **only** the LoRA adapter weights + head (~3 MB) instead of the full UNI encoder (~1.2 GB). Saves **~100 minutes of Drive I/O** across the 5-fold training.
- 💽 Per-epoch checkpoints write to local SSD (`/content/lora_ckpts_local`); a copy is mirrored to Drive every 3 epochs for cross-session resume.
- ⚡ bfloat16 autocast on A100 (drops `GradScaler` — bf16 has fp32 dynamic range, no scale-rejection retries).
- 🚀 `torch.set_float32_matmul_precision('high')` for TF32 matmul.
- 🎨 Image augmentation now runs on GPU (after H2D), not CPU. uint8 → GPU → float on GPU = 4× smaller transfer.
- 🧵 Stage 0 frozen-feature loading parallelized with 8 threads (Drive I/O is network-bound).

**Storage warning:**
- The Stage 1 extracted patches will use ~174 GB on local SSD AND ~174 GB on Drive (128 patches × 224² × 3 bytes × 9128 slides). Colab A100 has 235 GB SSD — workable but tight.

**Set this Colab Secret BEFORE running:** key icon (🔑) on left sidebar → add `HF_TOKEN`, toggle "Notebook access" on.


## 🧪 Smoke-Test Workflow (run this FIRST, with 200 slides)

The configuration cell below sets `TEST_MODE = True`, which makes every stage run on a 200-slide stratified sample (~40 per kfold). All outputs go to `*_test` directories so your real `PANDA_LoRA/` results are untouched.

**Recommended order:**
1. Set your `HF_TOKEN` (Colab Secret + the placeholder in cell 8).
2. Run all cells top-to-bottom with `TEST_MODE = True`. Expect Stage 0–4 to complete in well under an hour total on A100.
3. If everything finishes without errors and `Stage 4` prints sane QWKs, set `TEST_MODE = False` in the configuration cell and re-run from Stage 0. Production outputs go to `PANDA_LoRA/`, test outputs stay in `PANDA_LoRA_test/`, so the two never collide.

**What was fixed vs the original notebook:** Stage 0 now skips slides with no coordinates instead of fabricating zeros (which silently corrupted training data). Stage 3 extracts at 224 to match Stage 1 (no more train/inference distribution shift). The duplicate `chunked_encoder_forward` was removed. The Stage 2 SSD-copy probe now retries failed probe files in the main loop instead of silently dropping them. All `json.dump` calls go through `_to_jsonable` so numpy scalars don't crash the resume system on older numpy.

## 0. Environment Setup

## ⚠️ STEP 1 — Run this FIRST, then RESTART RUNTIME

The cell below upgrades `torchao` to a version compatible with peft. Colab's pre-installed `torchao` (0.10.0) breaks LoRA injection.

**After running this cell:**
1. Wait for the install to finish (~30 seconds)
2. **Runtime → Restart session** (NOT "Disconnect and delete" — local SSD files are preserved on restart)
3. Then run the rest of the notebook from the top


In [1]:
# Fix peft/torchao compatibility — REQUIRED before LoRA injection
# After this finishes, click: Runtime → Restart session
!pip install -q --upgrade "torchao>=0.16.0"
print()
print("=" * 60)
print("✓ torchao upgraded.")
print("✓ NOW DO: Runtime → Restart session (NOT 'Disconnect and delete')")
print("=" * 60)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.0 MB/s eta 0:00:00

✓ torchao upgraded.
✓ NOW DO: Runtime → Restart session (NOT 'Disconnect and delete')


In [2]:
# ── CLAM-TR LoRA v1: Environment Setup ──
import subprocess

# Install system dependencies
for pkg in ['openslide-tools', 'libgl1-mesa-glx']:
    subprocess.run(['apt-get', 'install', '-y', '-q', pkg], capture_output=True)

# Install Python packages (peft for LoRA)
for pkg in ['openslide-python', 'timm', 'huggingface_hub', 'h5py',
            'opencv-python-headless', 'peft', 'psutil']:
    subprocess.run(['pip', 'install', '-q', pkg], capture_output=True)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete.")

Mounted at /content/drive
Setup complete.


## 1. Imports

In [3]:
# ── Imports ──
import os
import json
import time
import h5py
import threading
import queue
import shutil
import concurrent.futures
from contextlib import nullcontext
import numpy as np
import pandas as pd
from datetime import datetime
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint

from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from scipy import stats
import matplotlib.pyplot as plt

import timm
from torchvision import transforms
from huggingface_hub import hf_hub_download, login as hf_login
from tqdm import tqdm
from PIL import Image
import cv2

# LoRA
from peft import LoraConfig, get_peft_model
import random

# ── HuggingFace Auth (needed for MahmoodLab/UNI gated model) ──
# Get your token from: https://huggingface.co/settings/tokens
# Also request access at: https://huggingface.co/MahmoodLab/UNI
HF_TOKEN = "hf_YOUR_HF_TOKEN_HERE"  # <-- Paste your token (rotate the old leaked one first!)
hf_login(token=HF_TOKEN)
print("HuggingFace: Logged in")

# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False  # SPEED: allow non-deterministic for perf
    torch.backends.cudnn.benchmark = True        # SPEED: auto-tune conv algorithms (fixed input size)
    torch.backends.cuda.matmul.allow_tf32 = True # SPEED: TF32 for matmuls (A100 native)
    torch.backends.cudnn.allow_tf32 = True       # SPEED: TF32 for convolutions

# SPEED: set float32 matmul precision to 'high' for TF32 path on A100
torch.set_float32_matmul_precision('high')

# Detect bf16 capability (A100 supports it; T4/V100 do not)
BF16_OK = (
    device.type == 'cuda'
    and torch.cuda.is_bf16_supported()
)
print(f"bfloat16 supported: {BF16_OK}")

# ── JSON-safe helper (numpy.float64/int64 are not JSON-serializable on numpy<2.0) ──
def _to_jsonable(obj):
    """Recursively convert numpy/torch scalar types to Python natives so json.dump works."""
    if isinstance(obj, dict):
        return {k: _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if torch.is_tensor(obj) and obj.dim() == 0:
        return obj.item()
    return obj

print("Imports complete.")


HuggingFace: Logged in
Device: cuda
GPU: NVIDIA L4
Memory: 23.7 GB
bfloat16 supported: True
Imports complete.


## 2. Configuration

In [4]:
# ═══════════════════════════════════════════════════════════════
# 🧪 TEST MODE — flip to False once the smoke test passes
# ═══════════════════════════════════════════════════════════════
# When TEST_MODE is True the pipeline runs end-to-end on TEST_N_SLIDES (200)
# slides, stratified across the 5 kfolds, and writes ALL outputs to *_test
# directories so your production runs (PANDA_LoRA/) are untouched.
# After every stage finishes successfully, set TEST_MODE = False and re-run.
TEST_MODE = False
TEST_N_SLIDES = 200
TEST_SEED = 42

# ═══════════════════════════════════════════════════════════════
# 💾 SKIP_SSD_COPY — bypass the 24-hour Drive→SSD copy
# ═══════════════════════════════════════════════════════════════
# When True, Stage 2 streams patches directly from Drive into the 167 GB
# global RAM cache, skipping the multi-hour SSD copy entirely.
# Recommended on Colab A100 (High-RAM) where you have plenty of RAM and the
# Colab session is wall-clock limited to 12-24 hours.
# Set False only if you are running on a machine where Drive reads are slow
# but local SSD is fast (rare).
SKIP_SSD_COPY = True

# ═══════════════════════════════════════════════════════════════
# 🎯 FOLD_TO_RUN — single-fold-per-session mode (RECOMMENDED)
# ═══════════════════════════════════════════════════════════════
# Run ONE fold per Colab session. Each session ~3.5h, well under 24h wall.
# Workflow:
#   Session 1: FOLD_TO_RUN = 0 → saves lora_fold0_best.pt to Drive
#   Session 2: FOLD_TO_RUN = 1 → saves lora_fold1_best.pt
#   ... and so on through fold 4.
# Set FOLD_TO_RUN = None to attempt all 5 folds in one session (will hit 24h wall).
FOLD_TO_RUN = None  # ← change this each session: 0, 1, 2, 3, 4

# ═══════════════════════════════════════════════════════════════
# Paths (auto-suffixed with '_test' when TEST_MODE is on)
# ═══════════════════════════════════════════════════════════════
_SUFFIX = '_test' if TEST_MODE else ''

SLIDES_DIR = '/content/drive/MyDrive/PANDA_Subset_9128/images'                 # READ-ONLY (shared)
FROZEN_FEATURES_DIR = '/content/drive/MyDrive/UNI_features'                     # READ-ONLY (shared)
V20_CHECKPOINT_DIR = '/content/drive/MyDrive/CLAM_TR_Results/v20_fullscale/UNI/checkpoints'  # READ-ONLY
LABELS_PATH = '/content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/Kaggle-PANDA-1st-place-solution-master/input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv'

# LoRA output directories — all writes go under here, suffixed in test mode
LORA_BASE_DIR = f'/content/drive/MyDrive/PANDA_LoRA{_SUFFIX}'
PATCH_INDICES_DIR = os.path.join(LORA_BASE_DIR, 'patch_indices')
EXTRACTED_PATCHES_DIR = os.path.join(LORA_BASE_DIR, 'extracted_patches')
LORA_CHECKPOINTS_DIR = os.path.join(LORA_BASE_DIR, 'checkpoints')
LORA_FEATURES_DIR = os.path.join(LORA_BASE_DIR, 'lora_features')
LORA_RESULTS_DIR = os.path.join(LORA_BASE_DIR, 'results')
LOCAL_PATCHES_DIR = f'/content/local_patches{_SUFFIX}'   # local SSD for training I/O
LOCAL_CKPT_DIR    = f'/content/lora_ckpts_local{_SUFFIX}'  # local SSD for epoch checkpoints

for d in [LORA_BASE_DIR, PATCH_INDICES_DIR, EXTRACTED_PATCHES_DIR,
          LORA_CHECKPOINTS_DIR, LORA_FEATURES_DIR, LORA_RESULTS_DIR,
          LOCAL_PATCHES_DIR, LOCAL_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ═══════════════════════════════════════════════════════════════
# V20-compatible model config
# ═══════════════════════════════════════════════════════════════
V20_CONFIG = {
    'input_dim': 1024,
    'hidden_dim': 512,
    'n_classes': 6,
    'num_heads': 8,
    'K': 128,
    'ffn_expansion': 4,
    'num_groups': 8,
}

# V20 best HP set (used for SpatialMultRetNet)
V20_BEST_HP = {
    'attention_temperature': 0.7,
    'dropout': 0.15,
    'gamma_values': [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
}

# ═══════════════════════════════════════════════════════════════
# LoRA config
# ═══════════════════════════════════════════════════════════════
LORA_CONFIG = {
    'r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'target_modules': ['qkv'],
    'bias': 'none',
}

# ═══════════════════════════════════════════════════════════════
# Training config
# ═══════════════════════════════════════════════════════════════
TRAIN_CONFIG = {
    'lr_lora': 5e-5,
    'lr_head': 2e-4,
    'weight_decay': 0.01,
    'epochs': 10,
    'patience': 5,
    'early_stop_threshold': 0.001,
    'K': 128,                     # ALL 128 patches for training
    'chunk_size': 128,            # A100: single pass for all 128 patches
    'batch_size': 1,              # 1 slide per step (MIL)
    'accumulation_steps': 1,      # A100: direct step every slide (no accumulation overhead)
    'gradient_checkpointing': False,  # SPEED: OFF = ~40% faster (enough VRAM with LoRA 0.26%)
    'use_bf16': True,                 # SPEED: A100 bf16 — drop GradScaler, more stable than fp16
    'max_grad_norm': 1.0,
    'n_folds': 5,
    'seed': 42,
    'use_class_weights': True,
    # Aggregation head HP (match V20 best)
    'head_loss': 'focal',
    'head_focal_gamma': 1.0,
    'head_label_smoothing': 0.1,
    'head_dropout': 0.15,
    'head_attention_temperature': 0.7,
}

# Patch extraction config
# Save at 224x224 (UNI input size) to reduce storage: ~175GB vs ~228GB at 256
EXTRACTION_CONFIG = {
    'patch_size': 224,
    'target_magnification': 20,
    'top_k_patches': 128,
    'encoder_batch_size': 512,    # A100: max inference batch (40GB VRAM)
    'slide_ext': '.tiff',
}

def load_labels():
    """Load labels CSV with kfold 1-5 → 0-4 conversion.

    When TEST_MODE is True, returns a stratified subsample of TEST_N_SLIDES
    rows (~equal slides per fold) so each stage exercises the full code path
    on a tiny dataset. Sample is deterministic (seeded by TEST_SEED).
    """
    df = pd.read_csv(LABELS_PATH)
    if df['kfold'].min() == 1:
        print(f"  Converting kfold {df['kfold'].min()}-{df['kfold'].max()} -> 0-4")
        df['kfold'] = df['kfold'] - 1
    assert df['kfold'].min() == 0 and df['kfold'].max() == 4, (
        f"kfold range error: {df['kfold'].min()}-{df['kfold'].max()}")
    if TEST_MODE:
        n_folds = df['kfold'].nunique()
        n_per_fold = max(1, TEST_N_SLIDES // n_folds)
        df = (df.groupby('kfold', group_keys=False)
                .apply(lambda g: g.sample(min(len(g), n_per_fold), random_state=TEST_SEED))
                .reset_index(drop=True))
        print(f"  TEST_MODE: sampled {len(df)} slides ({n_per_fold}/fold across {n_folds} folds)")
        # Show class distribution so unbalanced folds are visible
        print(f"  Per-fold counts: {df.groupby('kfold').size().to_dict()}")
        print(f"  Per-grade counts: {df.groupby('isup_grade').size().to_dict()}")
    return df

print("Configuration loaded.")
print(f"  TEST_MODE: {TEST_MODE}  ({'sampling ' + str(TEST_N_SLIDES) + ' slides' if TEST_MODE else 'full 9128 slides'})")
print(f"  SKIP_SSD_COPY: {SKIP_SSD_COPY}  ({'streaming from Drive into RAM cache' if SKIP_SSD_COPY else 'will copy 175GB Drive→SSD first'})")
print(f"  FOLD_TO_RUN: {FOLD_TO_RUN}  ({'all 5 folds (will hit 24h wall)' if FOLD_TO_RUN is None else f'fold {FOLD_TO_RUN} only — single-session mode'})")
print(f"  Slides: {SLIDES_DIR}")
print(f"  Frozen features: {FROZEN_FEATURES_DIR}")
print(f"  V20 checkpoints: {V20_CHECKPOINT_DIR}")
print(f"  LoRA output: {LORA_BASE_DIR}")
print(f"  LoRA rank: {LORA_CONFIG['r']}, alpha: {LORA_CONFIG['lora_alpha']}")
print(f"  Training: K={TRAIN_CONFIG['K']}, chunk={TRAIN_CONFIG['chunk_size']}, "
      f"epochs={TRAIN_CONFIG['epochs']}, accum={TRAIN_CONFIG['accumulation_steps']}")

# Resource profile
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
disk_free_gb = shutil.disk_usage('/content').free / 1e9
print(f"\n  Resources:")
print(f"    RAM: {ram_gb:.0f} GB total, {psutil.virtual_memory().available / 1e9:.0f} GB free")
if torch.cuda.is_available():
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"    GPU: {gpu_gb:.1f} GB ({torch.cuda.get_device_name(0)})")
print(f"    Disk: {disk_free_gb:.0f} GB free")


Configuration loaded.
  TEST_MODE: False  (full 9128 slides)
  SKIP_SSD_COPY: True  (streaming from Drive into RAM cache)
  FOLD_TO_RUN: None  (all 5 folds (will hit 24h wall))
  Slides: /content/drive/MyDrive/PANDA_Subset_9128/images
  Frozen features: /content/drive/MyDrive/UNI_features
  V20 checkpoints: /content/drive/MyDrive/CLAM_TR_Results/v20_fullscale/UNI/checkpoints
  LoRA output: /content/drive/MyDrive/PANDA_LoRA
  LoRA rank: 8, alpha: 16
  Training: K=128, chunk=128, epochs=10, accum=1

  Resources:
    RAM: 57 GB total, 55 GB free
    GPU: 23.7 GB (NVIDIA L4)
    Disk: 207 GB free


## v8 -- sharded tar workflow (disconnect-resilient, no FUSE overflow)

**What changed vs v6/v7.** v7 tried to build one 174 GB tar and push it at the end -- fragile: a single disconnect or FUSE overflow loses 3 hours of work. v8 splits the same data into 18 smaller (~10 GB) shards, pushes each one as soon as it is built, and deletes the local copy. Each shard becomes permanent on Drive after ~11 min. If the runtime disconnects, the next session resumes from the next missing shard.

**What you must do.** Set `FOLD_TO_RUN` in the config cell above and run all cells. Same as v6 -- the new cell below handles everything automatically.

**Timing.**
- First session ever: ~3.5 h (18 shards x 11 min each) + ~17 min unpack + ~3-4 h Stage 2 = ~7-8 h.
- Every later session: ~17 min unpack + ~3-4 h Stage 2 = ~4-5 h.

**Disconnect prevention.** Before running, paste this into your browser DevTools console (F12 -> Console) and keep the Colab tab in the foreground:

```javascript
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
```

The cell is idempotent: re-running it skips shards already on Drive and skips unpacking if local SSD is full.

In [5]:
# v8 SHARDED TAR WORKFLOW -- automatic, idempotent, disconnect-resilient
# Builds 18 ~10 GB shards on Drive (incrementally), then unpacks all shards
# to LOCAL_PATCHES_DIR. Re-points EXTRACTED_PATCHES_DIR at the local copy so
# Stage 2 reads from SSD (~500 MB/s) instead of Drive (~20 MB/s).
#
# Resumability: shards already on Drive are skipped. /content/ never holds
# more than one shard, so FUSE buffer cannot overflow.

import os, time, subprocess, tarfile
import shutil as _sh_v8

# Capture original Drive-side paths from cell 11 globals
_SRC_DRIVE_PATCHES = os.path.join(LORA_BASE_DIR, 'extracted_patches')
SHARD_DIR_DRIVE    = os.path.join(LORA_BASE_DIR, 'patches_shards')
FILES_PER_SHARD    = 500   # ~10 GB at 19 MB per file

def _v8_size_gb(p):
    return os.path.getsize(p) / 1e9 if os.path.exists(p) else 0.0

def _v8_build_and_push_shards():
    if not os.path.isdir(_SRC_DRIVE_PATCHES):
        raise RuntimeError(f'Source patches dir missing: {_SRC_DRIVE_PATCHES}\n'
                           f'Run Stages 0 and 1 first.')
    all_files = sorted(f for f in os.listdir(_SRC_DRIVE_PATCHES) if f.endswith('.pt'))
    if not all_files:
        raise RuntimeError(f'No .pt files in {_SRC_DRIVE_PATCHES}')

    shards = [all_files[i:i+FILES_PER_SHARD]
              for i in range(0, len(all_files), FILES_PER_SHARD)]
    print(f'[v8] Source: {len(all_files)} files -> {len(shards)} shards '
          f'({FILES_PER_SHARD} files / ~10 GB each)')

    os.makedirs(SHARD_DIR_DRIVE, exist_ok=True)
    overall_t0 = time.time()

    for sidx, shard_files in enumerate(shards):
        name = f'patches.{sidx:03d}.tar'
        drv  = os.path.join(SHARD_DIR_DRIVE, name)
        loc  = f'/content/_v8_shard_{sidx:03d}.tar'

        # Resume: skip shards already on Drive (size > 1 GB sanity check)
        if os.path.exists(drv) and os.path.getsize(drv) > 1024**3:
            print(f'[{sidx+1:>2}/{len(shards)}] {name} already on Drive '
                  f'({_v8_size_gb(drv):.1f} GB) -- skip')
            continue
        if os.path.exists(drv):
            os.remove(drv)

        # --- Build local shard ---
        print(f'[{sidx+1:>2}/{len(shards)}] building {name} '
              f'({len(shard_files)} files)...', flush=True)
        if os.path.exists(loc):
            os.remove(loc)
        t0 = time.time()
        failed = []
        with tarfile.open(loc, 'w') as tar:
            for f in shard_files:
                for attempt in range(3):
                    try:
                        tar.add(os.path.join(_SRC_DRIVE_PATCHES, f), arcname=f)
                        break
                    except (OSError, IOError):
                        if attempt == 2:
                            failed.append(f)
                        else:
                            time.sleep(2)
        sz = os.path.getsize(loc)
        build_min = (time.time() - t0) / 60
        print(f'  built {sz/1e9:.2f} GB in {build_min:.1f}m '
              f'(failed={len(failed)})', flush=True)

        # --- Push to Drive with fsync (caps FUSE buffer at one chunk) ---
        print(f'  pushing with fsync...', flush=True)
        t0 = time.time()
        written = 0
        CHUNK = 64 * 1024 * 1024
        with open(loc, 'rb') as fsrc, open(drv, 'wb') as fdst:
            fd = fdst.fileno()
            while True:
                buf = fsrc.read(CHUNK)
                if not buf:
                    break
                for attempt in range(6):
                    try:
                        fdst.write(buf)
                        fdst.flush()
                        os.fsync(fd)
                        break
                    except (OSError, IOError) as e:
                        if attempt == 5:
                            raise
                        time.sleep(15 * (attempt + 1))
                written += len(buf)
        push_min = (time.time() - t0) / 60
        rate = (written/1e6) / (push_min*60 + 1e-9)
        print(f'  pushed in {push_min:.1f}m ({rate:.0f} MB/s)', flush=True)

        assert os.path.getsize(drv) == sz, f'size mismatch on {name}'
        os.remove(loc)

        # Overall progress
        done = sidx + 1
        elapsed_m = (time.time() - overall_t0) / 60
        eta_m = elapsed_m * (len(shards) - done) / max(done, 1)
        print(f'  >>> overall: {done}/{len(shards)} shards | '
              f'{elapsed_m:.0f}m elapsed | ETA {eta_m:.0f}m', flush=True)

    print(f'[v8] All {len(shards)} shards on Drive.')
    return len(shards)

def _v8_unpack_all_shards():
    os.makedirs(LOCAL_PATCHES_DIR, exist_ok=True)
    src_count = sum(1 for f in os.listdir(_SRC_DRIVE_PATCHES) if f.endswith('.pt'))
    n_existing = sum(1 for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt'))

    if n_existing >= int(src_count * 0.99):
        print(f'[unpack] local SSD has {n_existing}/{src_count} files -- skip')
        return

    if n_existing > 0:
        print(f'[unpack] local SSD has {n_existing}/{src_count}, completing unpack')

    shard_files = sorted(f for f in os.listdir(SHARD_DIR_DRIVE) if f.endswith('.tar'))
    print(f'[unpack] {len(shard_files)} shards to unpack to {LOCAL_PATCHES_DIR}')
    overall_t0 = time.time()
    for i, name in enumerate(shard_files, 1):
        drv = os.path.join(SHARD_DIR_DRIVE, name)
        t0 = time.time()
        subprocess.run(['tar', '-xf', drv, '-C', LOCAL_PATCHES_DIR], check=True)
        print(f'  [{i:>2}/{len(shard_files)}] unpacked {name} '
              f'in {(time.time()-t0)/60:.1f}m', flush=True)
    print(f'[unpack] DONE in {(time.time()-overall_t0)/60:.1f}m')


_v8_build_and_push_shards()
_v8_unpack_all_shards()

# Re-point Stage 2/3/4 at the local SSD copy. With SKIP_SSD_COPY=True
# (default in v6), run_stage_2() reads patches from EXTRACTED_PATCHES_DIR;
# pointing it at /content/local_patches/ delivers SSD-speed I/O without any
# other code change.
EXTRACTED_PATCHES_DIR = LOCAL_PATCHES_DIR
n_ready = sum(1 for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt'))
print(f'\n[v8] READY: {n_ready} patches on local SSD at {LOCAL_PATCHES_DIR}')
print(f'[v8] EXTRACTED_PATCHES_DIR -> {EXTRACTED_PATCHES_DIR}')
print(f'[v8] Stage 2 will stream from local SSD (~500 MB/s) instead of Drive')


[v8] Source: 9128 files -> 19 shards (500 files / ~10 GB each)
[ 1/19] patches.000.tar already on Drive (9.5 GB) -- skip
[ 2/19] patches.001.tar already on Drive (9.5 GB) -- skip
[ 3/19] patches.002.tar already on Drive (9.5 GB) -- skip
[ 4/19] patches.003.tar already on Drive (9.5 GB) -- skip
[ 5/19] patches.004.tar already on Drive (9.5 GB) -- skip
[ 6/19] patches.005.tar already on Drive (9.5 GB) -- skip
[ 7/19] patches.006.tar already on Drive (9.5 GB) -- skip
[ 8/19] patches.007.tar already on Drive (9.6 GB) -- skip
[ 9/19] patches.008.tar already on Drive (9.6 GB) -- skip
[10/19] patches.009.tar already on Drive (9.6 GB) -- skip
[11/19] patches.010.tar already on Drive (9.6 GB) -- skip
[12/19] patches.011.tar already on Drive (9.5 GB) -- skip
[13/19] patches.012.tar already on Drive (9.6 GB) -- skip
[14/19] patches.013.tar already on Drive (9.5 GB) -- skip
[15/19] patches.014.tar already on Drive (9.5 GB) -- skip
[16/19] patches.015.tar already on Drive (9.5 GB) -- skip
[17/19] p

## 3. V20 Model Components (Exact Copy)

In [6]:
# ============================================================================
# V20 Model Components — exact copy for checkpoint loading & attention extraction
# ============================================================================

class GatedAttention(nn.Module):
    """V7 exact. Gated attention with temperature scaling."""
    def __init__(self, input_dim=512, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout))
        self.attention_b = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_c = nn.Linear(hidden_dim, 1)

    def forward(self, x, temperature=1.0):
        a = self.attention_a(x)
        b = self.attention_b(x)
        return self.attention_c(a * b).squeeze(-1) / temperature


class SwishGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        return x * torch.sigmoid(self.proj(x))


class FFNBlock(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        hidden = dim * expansion
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return x + self.net(self.norm(x))


class RetentionHead(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadMult(nn.Module):
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale * D
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadAdd(nn.Module):
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = scores + torch.log(D + 1e-6)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionMult(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadMult(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [head(x, coords) for head in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class SpatialRetentionAdd(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadAdd(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [head(x, coords) for head in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class RetentionBlock(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None,
                 use_spatial=False, spatial_mode='multiplicative',
                 use_norm=True, norm_type='group', num_groups=8,
                 use_residual=True, use_gate=True, use_ffn=True,
                 ffn_expansion=4, dropout=0.25):
        super().__init__()
        self.use_spatial = use_spatial
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.use_gate = use_gate
        self.use_ffn = use_ffn

        if use_spatial:
            assert gamma_values is not None and len(gamma_values) == num_heads
            HeadClass = (SpatialRetentionHeadMult if spatial_mode == 'multiplicative'
                         else SpatialRetentionHeadAdd)
            self.heads = nn.ModuleList([
                HeadClass(hidden_dim, gamma_values[i], num_heads)
                for i in range(num_heads)])
        else:
            self.heads = nn.ModuleList([
                RetentionHead(hidden_dim, num_heads) for _ in range(num_heads)])

        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        if use_norm:
            self.norm = (nn.GroupNorm(num_groups, hidden_dim) if norm_type == 'group'
                         else nn.LayerNorm(hidden_dim))
        if use_gate:
            self.gate = SwishGate(hidden_dim)
        if use_ffn:
            self.ffn = FFNBlock(hidden_dim, ffn_expansion, dropout)

    def forward(self, x, coords=None):
        identity = x
        if self.use_spatial:
            head_outputs = [head(x, coords) for head in self.heads]
        else:
            head_outputs = [head(x) for head in self.heads]

        out = self.output_proj(torch.cat(head_outputs, dim=-1))
        out = self.dropout(out)

        if self.use_norm:
            out = self.norm(out)
        if self.use_residual:
            out = out + identity
        if self.use_gate:
            out = self.gate(out)
        if self.use_ffn:
            out = self.ffn(out)
        return out


class ABMIL(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25):
        super().__init__()
        self.needs_coords = False
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = F.softmax(self.attention(h, temperature), dim=0)
        M = torch.sum(A.unsqueeze(-1) * h, dim=0)
        return self.classifier(M)


class DualBranchModel(nn.Module):
    """V20 exact DualBranchModel."""
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25,
                 K=128, retention_module=None, needs_coords=False):
        super().__init__()
        self.K = K
        self.needs_coords = needs_coords
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.retention_module = retention_module
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = self.attention(h, temperature)
        A_softmax = F.softmax(A, dim=0)

        K_actual = min(self.K, h.size(0))
        topk_indices = torch.topk(A, K_actual).indices
        h_topk = h[topk_indices]
        A_topk = A_softmax[topk_indices]

        M_clam = torch.sum(A_topk.unsqueeze(-1) * h_topk, dim=0)

        if self.retention_module is None:
            M_retention = h_topk.mean(dim=0)
        elif self.needs_coords:
            coords_topk = coords[topk_indices]
            M_retention = self.retention_module(h_topk, coords_topk).mean(dim=0)
        else:
            M_retention = self.retention_module(h_topk).mean(dim=0)

        return self.classifier(M_clam + M_retention)

    def get_attention_scores(self, x, coords=None, temperature=1.0):
        """Return raw attention scores for all patches (for patch selection)."""
        h = self.feature_transform(x)
        A = self.attention(h, temperature)
        return A


def create_v20_model(experiment_name, hp_name='best'):
    """Create V20 model for checkpoint loading."""
    hp = V20_BEST_HP.copy()
    dropout = hp['dropout']
    gamma_values = hp['gamma_values']
    c = V20_CONFIG

    if experiment_name == 'SpatialMultRetNet':
        retention = RetentionBlock(
            hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
            gamma_values=gamma_values,
            use_spatial=True, spatial_mode='multiplicative',
            use_norm=True, norm_type='group', num_groups=c['num_groups'],
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=c['ffn_expansion'], dropout=dropout)
        model = DualBranchModel(
            c['input_dim'], c['hidden_dim'], c['n_classes'], dropout, c['K'],
            retention_module=retention, needs_coords=True)
    elif experiment_name == 'ABMIL':
        model = ABMIL(c['input_dim'], c['hidden_dim'], c['n_classes'], dropout)
    elif experiment_name == 'RetNet':
        retention = RetentionBlock(
            hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
            use_spatial=False,
            use_norm=True, norm_type='group', num_groups=c['num_groups'],
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=c['ffn_expansion'], dropout=dropout)
        model = DualBranchModel(
            c['input_dim'], c['hidden_dim'], c['n_classes'], dropout, c['K'],
            retention_module=retention, needs_coords=False)
    else:
        raise ValueError(f"Unknown experiment: {experiment_name}")

    return model

print("V20 model components defined.")

V20 model components defined.


## 3.1 Loss Functions & Metrics

In [7]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, n_classes=6, smoothing=0.1, weight=None):
        super().__init__()
        self.n_classes = n_classes
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, pred, target):
        if pred.dim() == 1:
            pred = pred.unsqueeze(0)
        if target.dim() == 0:
            target = target.unsqueeze(0)
        conf = 1.0 - self.smoothing
        smooth = self.smoothing / (self.n_classes - 1)
        one_hot = torch.zeros_like(pred).scatter_(1, target.unsqueeze(1), 1)
        smooth_target = one_hot * conf + (1 - one_hot) * smooth
        log_probs = F.log_softmax(pred, dim=1)
        loss = -(smooth_target * log_probs).sum(dim=1)
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()


class FocalLoss(nn.Module):
    def __init__(self, n_classes=6, gamma=1.0, smoothing=0.1, weight=None):
        super().__init__()
        self.n_classes = n_classes
        self.gamma = gamma
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, pred, target):
        if pred.dim() == 1:
            pred = pred.unsqueeze(0)
        if target.dim() == 0:
            target = target.unsqueeze(0)
        conf = 1.0 - self.smoothing
        smooth = self.smoothing / (self.n_classes - 1)
        one_hot = torch.zeros_like(pred).scatter_(1, target.unsqueeze(1), 1)
        smooth_target = one_hot * conf + (1 - one_hot) * smooth
        log_probs = F.log_softmax(pred, dim=1)
        probs = torch.exp(log_probs)
        pt = probs.gather(1, target.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma
        loss = -(smooth_target * log_probs).sum(dim=1) * focal_weight
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()


def create_loss(class_weights_tensor):
    """Create loss based on TRAIN_CONFIG (matches V20 best HP)."""
    weight = class_weights_tensor if TRAIN_CONFIG['use_class_weights'] else None
    if TRAIN_CONFIG['head_loss'] == 'focal':
        return FocalLoss(V20_CONFIG['n_classes'], TRAIN_CONFIG['head_focal_gamma'],
                         TRAIN_CONFIG['head_label_smoothing'], weight)
    else:
        return LabelSmoothingLoss(V20_CONFIG['n_classes'],
                                  TRAIN_CONFIG['head_label_smoothing'], weight)


def compute_detailed_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    acc = accuracy_score(y_true, y_pred)
    per_grade_acc = {}
    for g in range(6):
        mask = (y_true == g)
        per_grade_acc[str(g)] = float((y_pred[mask] == g).mean()) if mask.sum() > 0 else 0.0
    errors = y_pred.astype(float) - y_true.astype(float)
    return {
        'qwk': float(qwk),
        'grading_accuracy': float(acc),
        'per_grade_accuracy': per_grade_acc,
        'mae': float(np.abs(errors).mean()),
        'within_1': float((np.abs(errors) <= 1).mean()),
    }

print("Loss functions & metrics defined.")

Loss functions & metrics defined.


## 4. Stage 0 — Attention-Based Patch SelectionLoad the best V20 SpatialMultRetNet checkpoint, run frozen features through it,extract attention scores, and save the Top-128 patch indices per slide.

In [8]:
def run_stage_0():
    """Stage 0: Attention-based patch selection using V20 checkpoint."""
    output_path = os.path.join(PATCH_INDICES_DIR, 'top128_indices.json')
    partial_path = output_path + '.partial'

    # Check if already done
    if os.path.exists(output_path):
        with open(output_path, 'r') as f:
            existing = json.load(f)
        print(f"Stage 0 already complete: {len(existing)} slides indexed")
        return existing

    print("=" * 60)
    print("STAGE 0: Attention-Based Patch Selection")
    print("=" * 60)

    # Load labels
    df = load_labels()
    all_slide_ids = df['image_id'].tolist()
    print(f"Total slides: {len(all_slide_ids)}")

    # Resume from partial progress if available
    patch_indices_done = {}
    if os.path.exists(partial_path):
        try:
            with open(partial_path, 'r') as f:
                patch_indices_done = json.load(f)
            print(f"  Resuming from partial progress: {len(patch_indices_done)} slides already indexed")
        except (json.JSONDecodeError, IOError):
            patch_indices_done = {}

    # Find best V20 SpatialMultRetNet checkpoint
    # Look for fold results to find best fold
    ckpt_files = [f for f in os.listdir(V20_CHECKPOINT_DIR)
                  if 'SpatialMultRetNet_bestHP' in f and f.endswith('_epoch.pt')]

    # Also check experiment_progress.json for completed results
    progress_path = os.path.join(V20_CHECKPOINT_DIR, 'experiment_progress.json')
    best_fold = 0
    if os.path.exists(progress_path):
        with open(progress_path, 'r') as f:
            progress = json.load(f)
        for key in progress:
            if 'SpatialMultRetNet_bestHP' in key:
                results = progress[key].get('results', {})
                fold_qwks = results.get('fold_qwks', [])
                if fold_qwks:
                    best_fold = int(np.argmax(fold_qwks))
                    print(f"Best fold: {best_fold} (QWK={fold_qwks[best_fold]:.4f})")
                break

    # Load checkpoint — try epoch checkpoint first, then fold checkpoint
    ckpt_loaded = False
    model = create_v20_model('SpatialMultRetNet', 'best')

    # Try epoch checkpoint (has model_state_dict key)
    for pattern in [f'SpatialMultRetNet_bestHP_fold{best_fold}_epoch.pt',
                    f'SpatialMultRetNet_bestHP_fold{best_fold}.pt']:
        ckpt_path = os.path.join(V20_CHECKPOINT_DIR, pattern)
        if os.path.exists(ckpt_path):
            print(f"Loading checkpoint: {pattern}")
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
                model.load_state_dict(ckpt['model_state_dict'])
            elif isinstance(ckpt, dict) and 'best_model_state_dict' in ckpt:
                model.load_state_dict(ckpt['best_model_state_dict'])
            else:
                model.load_state_dict(ckpt)
            ckpt_loaded = True
            del ckpt
            break

    if not ckpt_loaded:
        # List available checkpoints for debugging
        available = os.listdir(V20_CHECKPOINT_DIR)
        print(f"Available files: {available[:20]}")
        raise FileNotFoundError("Could not find SpatialMultRetNet checkpoint!")

    model = model.to(device).eval()
    print(f"V20 model loaded (fold {best_fold})")

    # Process each slide (skip already-indexed ones)
    top_k = EXTRACTION_CONFIG['top_k_patches']
    patch_indices = dict(patch_indices_done)  # start from partial progress
    to_process = [sid for sid in all_slide_ids if sid not in patch_indices]
    print(f"Slides to process: {len(to_process)} (skipping {len(patch_indices)} already done)")

    t0 = time.time()
    missing = 0
    new_count = 0

    # SPEED: parallel feature loading from Drive (network-bound, threading gives ~6-8x)
    def _load_feat(sid):
        feat_path = os.path.join(FROZEN_FEATURES_DIR, f'{sid}.pt')
        if not os.path.exists(feat_path):
            return sid, None, None
        try:
            data = torch.load(feat_path, map_location='cpu', weights_only=False)
            if isinstance(data, dict):
                features = data.get('features', data.get('embeddings', list(data.values())[0]))
                if isinstance(features, np.ndarray):
                    features = torch.tensor(features, dtype=torch.float32)
                else:
                    features = features.float()
                if features.ndim == 3:
                    features = features.squeeze(0)
                coords = data.get('coords', data.get('coordinates'))
                if coords is None:
                    # FIX: fail the slide instead of fabricating (0, 0) coords.
                    # Otherwise Stage 1 would extract the same patch 128 times
                    # and silently corrupt training data.
                    return sid, None, None
                if isinstance(coords, np.ndarray):
                    coords = torch.tensor(coords, dtype=torch.float32)
                else:
                    coords = coords.float()
                if coords.ndim == 3:
                    coords = coords.squeeze(0)
            else:
                # Raw tensor — no coords stored. Fail consistent with the
                # dict-with-no-coords branch above.
                return sid, None, None
            return sid, features, coords
        except Exception:
            return sid, None, None

    LOAD_WORKERS = 8  # Drive I/O is network-bound; threading helps a lot

    with concurrent.futures.ThreadPoolExecutor(max_workers=LOAD_WORKERS) as ex:
        futures = {ex.submit(_load_feat, sid): sid for sid in to_process}
        pbar = tqdm(concurrent.futures.as_completed(futures),
                    total=len(futures), desc="Extracting attention")
        for fut in pbar:
            sid = futures[fut]
            _, features, coords = fut.result()
            if features is None:
                missing += 1
                continue

            n_patches = features.shape[0]
            k_actual = min(top_k, n_patches)

            with torch.no_grad():
                features_gpu = features.to(device, non_blocking=True)
                coords_gpu = coords.to(device, non_blocking=True)
                attn_scores = model.get_attention_scores(
                    features_gpu, coords_gpu, V20_BEST_HP['attention_temperature'])
                topk_result = torch.topk(attn_scores, k_actual)
                indices = topk_result.indices.cpu().tolist()
                scores = topk_result.values.cpu().tolist()

            selected_coords = coords[indices].tolist()
            patch_indices[sid] = {
                'indices': indices,
                'coords': selected_coords,
                'scores': scores,
                'n_total': n_patches,
                'n_selected': k_actual,
            }
            new_count += 1
            del features, coords, features_gpu, coords_gpu, attn_scores

            # Save partial progress every 500 slides (every 50 in test mode)
            _save_every = 50 if TEST_MODE else 500
            if new_count > 0 and new_count % _save_every == 0:
                with open(partial_path, 'w') as f:
                    json.dump(_to_jsonable(patch_indices), f)
                elapsed = time.time() - t0
                pbar.set_postfix(done=new_count, missing=missing,
                                 t=f"{elapsed/60:.1f}min")

    # Save final output
    with open(output_path, 'w') as f:
        json.dump(_to_jsonable(patch_indices), f)

    # Clean up partial file
    if os.path.exists(partial_path):
        os.remove(partial_path)

    elapsed = time.time() - t0
    print(f"\nStage 0 complete: {len(patch_indices)} slides in {elapsed/60:.1f}min")
    if missing > 0:
        print(f"  WARNING: {missing} slides had no frozen features")

    # Summary statistics
    n_selected = [v['n_selected'] for v in patch_indices.values()]
    print(f"  Patches per slide: min={min(n_selected)}, "
          f"max={max(n_selected)}, mean={np.mean(n_selected):.0f}")

    del model
    torch.cuda.empty_cache()
    return patch_indices

# Run Stage 0
patch_indices = run_stage_0()


Stage 0 already complete: 9128 slides indexed


## 5. Stage 1 — Patch Pre-Extraction from .tiff SlidesRead the Top-128 patches from raw .tiff files and save as uint8 tensors.This eliminates .tiff I/O during training.

In [9]:
import openslide

class SlidePreprocessor:
    """Extract specific patches from WSI at target magnification."""

    def __init__(self, patch_size=256, target_mag=20):
        self.patch_size = patch_size
        self.target_mag = target_mag

    def get_extraction_params(self, slide):
        """Find best pyramid level for target magnification."""
        try:
            obj_power = float(slide.properties.get('openslide.objective-power', 20))
        except (ValueError, KeyError):
            obj_power = 20

        downsample_needed = obj_power / self.target_mag
        best_level, best_diff = 0, float('inf')
        for level in range(slide.level_count):
            diff = abs(slide.level_downsamples[level] - downsample_needed)
            if diff < best_diff:
                best_diff, best_level = diff, level

        level_ds = slide.level_downsamples[best_level]
        actual_mag = obj_power / level_ds
        mag_ratio = actual_mag / self.target_mag
        read_size = int(round(self.patch_size * mag_ratio))
        step_l0 = int(round(read_size * level_ds))

        return best_level, read_size, step_l0, actual_mag

    def extract_patch_at_coord(self, slide, coord, level, read_size):
        """Extract a single patch at given level-0 coordinate."""
        x, y = int(coord[0]), int(coord[1])
        patch_pil = slide.read_region((x, y), level, (read_size, read_size)).convert('RGB')
        if read_size != self.patch_size:
            patch_pil = patch_pil.resize((self.patch_size, self.patch_size), Image.LANCZOS)
        return np.array(patch_pil, dtype=np.uint8)

print("SlidePreprocessor defined.")

SlidePreprocessor defined.


In [10]:
def _extract_one_slide(args):
    """Worker function: extract patches from one slide (runs in thread pool)."""
    sid, slide_path, coords, patch_size, target_mag, output_dir = args

    try:
        preprocessor = SlidePreprocessor(patch_size=patch_size, target_mag=target_mag)
        slide = openslide.OpenSlide(slide_path)
        level, read_size, _, _ = preprocessor.get_extraction_params(slide)

        patches = []
        valid_coords = []
        for coord in coords:
            try:
                patch = preprocessor.extract_patch_at_coord(slide, coord, level, read_size)
                patches.append(patch)
                valid_coords.append(coord)
            except Exception:
                continue

        slide.close()

        if len(patches) == 0:
            return sid, False, 'no valid patches'

        patches_tensor = torch.from_numpy(np.stack(patches))  # uint8
        coords_tensor = torch.tensor(valid_coords, dtype=torch.float32)

        output_path = os.path.join(output_dir, f'{sid}.pt')
        torch.save({
            'patches': patches_tensor,
            'coords': coords_tensor,
            'n_patches': len(patches),
        }, output_path)

        return sid, True, None

    except Exception as e:
        return sid, False, str(e)


def run_stage_1(patch_indices):
    """Stage 1: Pre-extract Top-128 patches from .tiff slides.

    Uses ThreadPoolExecutor for parallel extraction — Drive I/O is network-bound
    so threading gives ~6-8× speedup over sequential.
    """

    # Check progress
    progress_path = os.path.join(EXTRACTED_PATCHES_DIR, 'extraction_progress.json')
    if os.path.exists(progress_path):
        with open(progress_path, 'r') as f:
            progress = json.load(f)
        completed = set(progress.get('completed', []))
    else:
        completed = set()

    # Also check for existing .pt files
    existing = {f.replace('.pt', '') for f in os.listdir(EXTRACTED_PATCHES_DIR)
                if f.endswith('.pt')}
    completed = completed | existing

    to_process = [sid for sid in patch_indices if sid not in completed]

    print("=" * 60)
    print("STAGE 1: Patch Pre-Extraction from .tiff")
    print("=" * 60)
    print(f"Total slides: {len(patch_indices)}")
    print(f"Already extracted: {len(completed)}")
    print(f"To process: {len(to_process)}")

    if not to_process:
        print("All patches already extracted!")
        return

    patch_size = EXTRACTION_CONFIG['patch_size']
    target_mag = EXTRACTION_CONFIG['target_magnification']

    # Build work items — skip missing slides upfront
    work_items = []
    missing = 0
    for sid in to_process:
        slide_path = os.path.join(SLIDES_DIR, f'{sid}.tiff')
        if not os.path.exists(slide_path):
            missing += 1
            continue
        coords = patch_indices[sid]['coords']
        work_items.append((sid, slide_path, coords, patch_size, target_mag, EXTRACTED_PATCHES_DIR))
    if missing:
        print(f"  Skipping {missing} slides (file not found)")

    t0 = time.time()
    extracted = 0
    failed = []
    completed_list = list(completed)

    # Parallel extraction: 8 threads (Drive I/O is network-bound, threading helps ~6-8×)
    NUM_WORKERS = 8
    print(f"  Using {NUM_WORKERS} parallel workers")

    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(_extract_one_slide, item): item[0]
                   for item in work_items}

        pbar = tqdm(concurrent.futures.as_completed(futures),
                    total=len(futures), desc="Extracting patches")
        for future in pbar:
            sid = futures[future]
            sid_result, success, error = future.result()
            if success:
                extracted += 1
                completed_list.append(sid_result)
            else:
                failed.append((sid_result, error))

            done = extracted + len(failed)
            if done % 500 == 0 and done > 0:
                with open(progress_path, 'w') as f:
                    json.dump({
                        'completed': completed_list,
                        'failed': [(s, e) for s, e in failed],
                        'timestamp': datetime.now().isoformat()
                    }, f)
                elapsed = time.time() - t0
                rate = done / (elapsed / 60)
                remaining = (len(work_items) - done) / max(rate, 0.1)
                pbar.set_postfix(ok=extracted, fail=len(failed),
                                 rate=f"{rate:.0f}/min", eta=f"{remaining:.0f}min")

    # Final save
    with open(progress_path, 'w') as f:
        json.dump({
            'completed': completed_list,
            'failed': [(s, e) for s, e in failed],
            'timestamp': datetime.now().isoformat()
        }, f)

    elapsed = time.time() - t0
    print(f"\nStage 1 complete: {extracted} slides in {elapsed/60:.1f}min")
    if failed:
        print(f"  Failed: {len(failed)}")
        for sid, err in failed[:10]:
            print(f"    - {sid}: {err}")

    # Storage estimate
    total_bytes = sum(
        os.path.getsize(os.path.join(EXTRACTED_PATCHES_DIR, f))
        for f in os.listdir(EXTRACTED_PATCHES_DIR) if f.endswith('.pt'))
    print(f"  Storage used: {total_bytes / 1e9:.1f} GB")

# Run Stage 1
run_stage_1(patch_indices)

STAGE 1: Patch Pre-Extraction from .tiff
Total slides: 9128
Already extracted: 9128
To process: 0
All patches already extracted!


## 🔎 Sanity Check — STOP HERE before running Stage 2

Before burning ~10 hours on Stage 2 training, verify the Top-128 patches actually contain tissue (not white background or slide edges). If V20 attention picked garbage patches, Stage 2 will train on noise and you'll never know.

**Run the cell below.** If patches look like real prostate tissue → continue to Stage 2. If they look mostly white/empty → stop and debug V20 checkpoint loading.

In [11]:
# ── Sanity: visualize Top-128 patches on 3 random slides ──
import random
import matplotlib.pyplot as plt

sample_sids = random.sample(list(patch_indices.keys()), 3)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, sid in enumerate(sample_sids):
    pt_path = os.path.join(EXTRACTED_PATCHES_DIR, f'{sid}.pt')
    if not os.path.exists(pt_path):
        print(f"  Skipping {sid} — not yet extracted")
        for col in range(4):
            axes[row, col].axis('off')
        continue

    data = torch.load(pt_path, weights_only=False)
    patches = data['patches']        # (N, H, W, 3) uint8
    n = patches.shape[0]
    # Spread 4 indices across the available patches
    pick = [int(i * (n - 1) / 3) for i in range(4)] if n >= 4 else list(range(n))

    for col, idx in enumerate(pick):
        axes[row, col].imshow(patches[idx].numpy())
        axes[row, col].set_title(f"{sid[:10]}  patch #{idx}", fontsize=9)
        axes[row, col].axis('off')

plt.suptitle("Sanity: should look like prostate tissue, NOT white background", fontsize=12)
plt.tight_layout()
plt.show()

# Quick numeric check: mean pixel value per slide
# Tissue patches: mean ~140-200 (purple/pink). White background: mean ~240+.
print("\nMean pixel value per sampled slide (tissue ~140-200, background ~240+):")
for sid in sample_sids:
    pt_path = os.path.join(EXTRACTED_PATCHES_DIR, f'{sid}.pt')
    if os.path.exists(pt_path):
        data = torch.load(pt_path, weights_only=False)
        mean_val = data['patches'].float().mean().item()
        flag = "✓ tissue" if mean_val < 230 else "⚠ mostly background!"
        print(f"  {sid}: mean={mean_val:.1f}  {flag}")

Output hidden; open in https://colab.research.google.com to view.

## 6. Stage 2 — LoRA TrainingFine-tune UNI v1 with LoRA adapters + SpatialMultRetNet aggregation head.- Encoder processes patches in chunks of 32 with gradient checkpointing- Two-stage LR: slow for LoRA adapters, fast for aggregation head- 5-fold CV matching V20 fold splits

In [12]:
# ═══════════════════════════════════════════════════════════════
# UNI v1 Encoder with LoRA
# ═══════════════════════════════════════════════════════════════

def create_uni_lora_encoder():
    """Create UNI v1 encoder with LoRA adapters."""
    print("Loading UNI v1 (ViT-L/16)...")
    encoder = timm.create_model(
        "vit_large_patch16_224",
        img_size=224, patch_size=16,
        init_values=1e-5,
        num_classes=0,
        dynamic_img_size=True
    )
    weights_path = hf_hub_download("MahmoodLab/UNI", "pytorch_model.bin")
    encoder.load_state_dict(
        torch.load(weights_path, map_location="cpu", weights_only=False), strict=True)

    # Freeze all parameters
    for p in encoder.parameters():
        p.requires_grad = False

    # Conditional gradient checkpointing (SPEED: OFF by default for 2x faster backward)
    use_gc = TRAIN_CONFIG.get('gradient_checkpointing', False)
    if use_gc:
        encoder.set_grad_checkpointing(True)

    total_params = sum(p.numel() for p in encoder.parameters())
    gc_str = "ON" if use_gc else "OFF"
    print(f"UNI v1 loaded: {total_params:,} params (frozen, grad_ckpt={gc_str})")

    # Apply LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG['r'],
        lora_alpha=LORA_CONFIG['lora_alpha'],
        lora_dropout=LORA_CONFIG['lora_dropout'],
        target_modules=LORA_CONFIG['target_modules'],
        bias=LORA_CONFIG['bias'],
    )
    encoder = get_peft_model(encoder, lora_config)

    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    total = sum(p.numel() for p in encoder.parameters())
    assert trainable > 0, (
        f"LoRA matched zero modules! target_modules={LORA_CONFIG['target_modules']} "
        f"did not match any layers. Check module names with: "
        f"[n for n, _ in encoder.named_modules() if 'qkv' in n]"
    )
    print(f"LoRA applied: {trainable:,} trainable / {total:,} total "
          f"({100*trainable/total:.2f}%)")
    encoder.print_trainable_parameters()

    return encoder


# ═══════════════════════════════════════════════════════════════
# Aggregation Head (fresh init each fold, V20 SpatialMultRetNet architecture)
# ═══════════════════════════════════════════════════════════════

def create_aggregation_head():
    """Create SpatialMultRetNet aggregation head (V20 architecture)."""
    c = V20_CONFIG
    hp = V20_BEST_HP

    retention = RetentionBlock(
        hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
        gamma_values=hp['gamma_values'],
        use_spatial=True, spatial_mode='multiplicative',
        use_norm=True, norm_type='group', num_groups=c['num_groups'],
        use_residual=True, use_gate=True, use_ffn=True,
        ffn_expansion=c['ffn_expansion'], dropout=hp['dropout'])

    head = DualBranchModel(
        c['input_dim'], c['hidden_dim'], c['n_classes'], hp['dropout'], c['K'],
        retention_module=retention, needs_coords=True)

    params = sum(p.numel() for p in head.parameters())
    print(f"Aggregation head: {params:,} params")
    return head

print("UNI + LoRA and aggregation head factories defined.")

UNI + LoRA and aggregation head factories defined.


In [13]:
# ═══════════════════════════════════════════════════════════════
# A100 80GB + 167GB RAM — Global Cache + CUDA Prefetcher
# ═══════════════════════════════════════════════════════════════

_GPU_MEAN = None
_GPU_STD = None

def _get_gpu_norm(device):
    global _GPU_MEAN, _GPU_STD
    if _GPU_MEAN is None or _GPU_MEAN.device != device:
        _GPU_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        _GPU_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    return _GPU_MEAN, _GPU_STD


# ═══ GLOBAL RAM CACHE — loaded ONCE, shared across ALL 5 folds ═══
_GLOBAL_PATCH_CACHE = {}
_CACHE_LOADED = False

def preload_global_cache(all_slide_ids, patches_dir, fallback_dir=None, max_ram_gb=150):
    """Load ~8100 slides into RAM ONCE. Shared across all 5 folds.

    167GB RAM → ~150GB for cache → ~8100 of 9128 slides.
    Remaining ~1000 slides lazy-load from SSD (still fast).
    Saves 48 min × 5 folds = 4 HOURS of per-fold Drive preload.
    """
    global _GLOBAL_PATCH_CACHE, _CACHE_LOADED
    if _CACHE_LOADED:
        print(f"Global cache: {len(_GLOBAL_PATCH_CACHE)} slides (already loaded)")
        return

    import psutil
    ram_avail = psutil.virtual_memory().available / 1e9
    print(f"Global RAM preload (up to {max_ram_gb:.0f} GB, {ram_avail:.0f} GB available)...")

    t0 = time.time()
    loaded, total_bytes, failed = 0, 0, 0

    # FAST PATH: threaded loader. Drive reads are I/O-bound so threads help a lot.
    # 8 workers cut preload time from ~3.7h → ~45 min on 9128 slides.
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import threading
    PRELOAD_WORKERS = 8
    cache_lock = threading.Lock()

    def _load_one(sid):
        path = os.path.join(patches_dir, f'{sid}.pt')
        if not os.path.exists(path) and fallback_dir:
            path = os.path.join(fallback_dir, f'{sid}.pt')
        try:
            data = torch.load(path, map_location='cpu', weights_only=False)
            return sid, data['patches'], data['coords'], None
        except Exception as e:
            return sid, None, None, str(e)

    pbar = tqdm(total=len(all_slide_ids), desc="Global RAM preload (threaded)")
    stop_flag = False
    with ThreadPoolExecutor(max_workers=PRELOAD_WORKERS) as pool:
        futures = {pool.submit(_load_one, sid): sid for sid in all_slide_ids}
        for fut in as_completed(futures):
            sid, patches, coords, err = fut.result()
            pbar.update(1)
            if stop_flag:
                continue  # drain remaining futures fast without storing
            if err is not None:
                failed += 1
                continue
            with cache_lock:
                if total_bytes / 1e9 >= max_ram_gb:
                    stop_flag = True
                    continue
                _GLOBAL_PATCH_CACHE[sid] = (patches, coords)
                total_bytes += patches.nelement() * patches.element_size()
                loaded += 1
                if loaded % 200 == 0:
                    pbar.set_postfix(cached=loaded, gb=f"{total_bytes/1e9:.1f}")
    pbar.close()
    if stop_flag:
        print(f"\n  Hit {max_ram_gb:.0f}GB limit at {loaded} slides (drained pool)")

    _CACHE_LOADED = True
    elapsed = time.time() - t0
    ram_used = psutil.virtual_memory().used / 1e9
    ram_total = psutil.virtual_memory().total / 1e9
    uncached = len(all_slide_ids) - loaded

    print(f"  Cached: {loaded}/{len(all_slide_ids)} slides ({total_bytes/1e9:.1f} GB) in {elapsed/60:.1f} min")
    if failed: print(f"  Failed: {failed}")
    if uncached > 0: print(f"  Uncached: {uncached} slides (will lazy-load from SSD)")
    print(f"  RAM: {ram_used:.1f} / {ram_total:.0f} GB")


class LoRAPatchDataset(Dataset):
    """Uses global RAM cache. Falls back to SSD → Drive for uncached slides."""

    def __init__(self, slide_ids, labels, patches_dir, subsample_k=None,
                 fallback_dir=None, preload=False):
        self.slide_ids = list(slide_ids)
        self.labels = list(labels)
        self.patches_dir = patches_dir
        self.fallback_dir = fallback_dir
        self.subsample_k = subsample_k
        # preload param ignored — global cache handles everything

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        label = self.labels[idx]

        if sid in _GLOBAL_PATCH_CACHE:
            patches, coords = _GLOBAL_PATCH_CACHE[sid]
        else:
            path = os.path.join(self.patches_dir, f'{sid}.pt')
            if not os.path.exists(path) and self.fallback_dir:
                path = os.path.join(self.fallback_dir, f'{sid}.pt')
            data = torch.load(path, map_location='cpu', weights_only=False)
            patches = data['patches']
            coords = data['coords']

        if self.subsample_k is not None and patches.shape[0] > self.subsample_k:
            indices = torch.randperm(patches.shape[0])[:self.subsample_k]
            patches = patches[indices]
            coords = coords[indices]

        return patches, coords, label, sid


def lora_collate_fn(batch):
    return batch[0]


def prepare_patches_for_encoder(patches, augment=False, device=None):
    """SPEED: keep uint8 on CPU until H2D, do float conversion + aug + normalize on GPU.

    Old path was permute/float/div on CPU then aug on CPU then H2D — costs ~75MB CPU
    work per slide. New path: H2D the tiny uint8 tensor, then everything on GPU.
    """
    if device is not None and device.type == 'cuda':
        # Move uint8 first (4x smaller than float32 → faster transfer)
        x = patches.to(device, non_blocking=True)
        # NCHW + float in one shot on GPU
        x = x.permute(0, 3, 1, 2).contiguous().float().mul_(1.0 / 255.0)
        if augment:
            if torch.rand(1).item() > 0.5: x = x.flip(-1)
            if torch.rand(1).item() > 0.5: x = x.flip(-2)
            if torch.rand(1).item() > 0.5: x = x.rot90(1, [-2, -1])
            if torch.rand(1).item() > 0.5:
                brightness = 1.0 + (torch.rand(1).item() - 0.5) * 0.2
                x = x.mul_(brightness).clamp_(0, 1)
        mean, std = _get_gpu_norm(device)
        x = x.sub_(mean).div_(std)
        return x
    # CPU fallback (unchanged behavior)
    x = patches.permute(0, 3, 1, 2).float().div_(255.0)
    if augment:
        if torch.rand(1).item() > 0.5: x = x.flip(-1)
        if torch.rand(1).item() > 0.5: x = x.flip(-2)
        if torch.rand(1).item() > 0.5: x = x.rot90(1, [-2, -1])
        if torch.rand(1).item() > 0.5:
            brightness = 1.0 + (torch.rand(1).item() - 0.5) * 0.2
            x = x.mul_(brightness).clamp_(0, 1)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (x - mean) / std


class CUDAPrefetcher:
    def __init__(self, loader, device, augment=False):
        self.loader = iter(loader)
        self.device = device
        self.augment = augment
        self.stream = torch.cuda.Stream()
        self._preload()

    def _preload(self):
        try:
            patches, coords, label, sid = next(self.loader)
            with torch.cuda.stream(self.stream):
                self.next_patches = prepare_patches_for_encoder(
                    patches, augment=self.augment, device=self.device)
                self.next_coords = coords.to(self.device, non_blocking=True)
                self.next_label = label
                self.next_sid = sid
        except StopIteration:
            self.next_patches = None

    def __iter__(self):
        return self

    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        if self.next_patches is None:
            raise StopIteration
        out = (self.next_patches, self.next_coords, self.next_label, self.next_sid)
        self._preload()
        return out

    def __len__(self):
        return len(self.loader)


print("Global RAM cache + CUDA prefetcher defined.")


Global RAM cache + CUDA prefetcher defined.


In [14]:
# ═══════════════════════════════════════════════════════════════
# LoRA Training Loop
# ═══════════════════════════════════════════════════════════════

class EarlyStopping:
    def __init__(self, patience=5, threshold=0.001):
        self.patience = patience
        self.threshold = threshold
        self.counter = 0
        self.best = None

    def __call__(self, score):
        if self.best is None or score > self.best + self.threshold:
            self.best = score
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


def chunked_encoder_forward(encoder, patches_gpu, chunk_size):
    """Forward patches through encoder in chunks.

    Per-block gradient checkpointing is handled inside the encoder
    via set_grad_checkpointing(True), so we just do simple forward here.

    Args:
        encoder: UNI v1 + LoRA model
        patches_gpu: (N, 3, 224, 224) on GPU
        chunk_size: number of patches per forward pass

    Returns:
        embeddings: (N, 1024) on GPU
    """
    n = patches_gpu.shape[0]
    all_embeddings = []

    for i in range(0, n, chunk_size):
        chunk = patches_gpu[i:i+chunk_size]
        emb = encoder(chunk)
        all_embeddings.append(emb)

    return torch.cat(all_embeddings, dim=0)


def _extract_trainable_state(encoder):
    """SPEED: extract ONLY the trainable (LoRA) params from the encoder.

    UNI v1 is 300M params (~1.2GB). LoRA only changes ~700K of them. We don't
    need to checkpoint the frozen base every epoch — that's ~99.7% wasted I/O.
    """
    out = {}
    sd = encoder.state_dict()
    for n, p in encoder.named_parameters():
        if p.requires_grad:
            # PEFT names trainable params with `lora_A`/`lora_B`/etc. — the
            # state_dict key matches the parameter name exactly here.
            if n in sd:
                out[n] = sd[n].detach().cpu().clone()
    return out


def _load_trainable_state(encoder, trainable_state):
    """Load only LoRA adapter params back into the encoder (strict=False)."""
    missing = encoder.load_state_dict(trainable_state, strict=False)
    # missing.unexpected_keys should be empty; missing_keys is the entire frozen base — ignore.
    return encoder


def train_one_fold_lora(encoder, head, fold, train_loader, val_loader,
                        class_weights_tensor, device):
    """Train one fold with LoRA encoder + aggregation head.

    A100 OPTIMIZATIONS:
      - bf16 autocast (no GradScaler needed; matches fp32 range)
      - per-epoch checkpoints save ONLY LoRA + head state (~3MB) to local SSD
      - best-only checkpoint synced to Drive at end of fold
    """
    cfg = TRAIN_CONFIG
    chunk_size = cfg['chunk_size']
    accum_steps = cfg['accumulation_steps']
    attn_temp = cfg['head_attention_temperature']

    # Separate parameter groups
    lora_params = [p for p in encoder.parameters() if p.requires_grad]
    head_params = list(head.parameters())

    optimizer = torch.optim.AdamW([
        {'params': lora_params, 'lr': cfg['lr_lora'], 'weight_decay': cfg['weight_decay']},
        {'params': head_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    ])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
    early_stop = EarlyStopping(cfg['patience'], cfg['early_stop_threshold'])

    criterion = create_loss(class_weights_tensor)

    # SPEED: bf16 on A100 — no GradScaler needed (bf16 has fp32 exponent range)
    use_amp = device.type == 'cuda'
    use_bf16 = use_amp and cfg.get('use_bf16', True) and BF16_OK
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and not use_bf16))

    print(f"  AMP: {'bf16 (no scaler)' if use_bf16 else ('fp16 + scaler' if use_amp else 'off')}")

    best_qwk = float('-inf')
    best_lora_state = None     # LoRA adapter weights only (~few MB)
    best_head_state = None     # full head weights (~few MB)
    best_epoch = 0

    # SPEED: epoch checkpoints go to local SSD, NOT Drive (no upload bottleneck)
    local_ckpt_dir = LOCAL_CKPT_DIR if 'LOCAL_CKPT_DIR' in globals() else '/content/lora_ckpts_local'
    os.makedirs(local_ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(local_ckpt_dir, f'lora_fold{fold}_epoch.pt')
    meta_path = os.path.join(local_ckpt_dir, f'lora_fold{fold}_meta.json')

    # Drive-side resume probe (in case the SSD was wiped between sessions)
    drive_resume_path = os.path.join(LORA_CHECKPOINTS_DIR, f'lora_fold{fold}_resume.pt')
    drive_meta_path   = os.path.join(LORA_CHECKPOINTS_DIR, f'lora_fold{fold}_resume_meta.json')

    start_epoch = 0
    if os.path.exists(ckpt_path) and os.path.exists(meta_path):
        print(f"  Resuming fold {fold} from local SSD...")
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        with open(meta_path, 'r') as f:
            meta = json.load(f)
    elif os.path.exists(drive_resume_path) and os.path.exists(drive_meta_path):
        print(f"  Resuming fold {fold} from Drive (SSD missing)...")
        ckpt = torch.load(drive_resume_path, map_location='cpu', weights_only=False)
        with open(drive_meta_path, 'r') as f:
            meta = json.load(f)
    else:
        ckpt = None

    if ckpt is not None:
        # Apply LoRA adapter weights + head weights (strict=False on encoder)
        _load_trainable_state(encoder, ckpt['lora_state'])
        head.load_state_dict(ckpt['head_state'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        else:
            for _ in range(meta['epoch'] + 1):
                scheduler.step()
        if 'scaler_state_dict' in ckpt and not use_bf16:
            try:
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            except Exception:
                pass
        start_epoch = meta['epoch'] + 1
        best_qwk = meta['best_qwk']
        best_epoch = meta['best_epoch']
        early_stop.counter = meta.get('es_counter', 0)
        early_stop.best = meta.get('es_best', None)
        if 'best_lora_state' in ckpt:
            best_lora_state = ckpt['best_lora_state']
            best_head_state = ckpt['best_head_state']
        print(f"  Resumed from epoch {start_epoch} (best QWK={best_qwk:.4f})")
        del ckpt

    n_train = len(train_loader)
    epoch_bar = tqdm(range(start_epoch, cfg['epochs']),
                     desc=f"  LoRA fold {fold}", leave=True)

    for epoch in epoch_bar:
        epoch_start = time.time()

        # ── Train ──
        encoder.train(); head.train()
        epoch_loss = 0.0
        n_batches = 0
        optimizer.zero_grad(set_to_none=True)

        prefetcher = CUDAPrefetcher(train_loader, device, augment=True)
        _last_heartbeat = time.time()  # for keep-alive prints during training
        for batch_idx, (patches_gpu, coords_gpu, label, sid) in enumerate(prefetcher):
            if batch_idx == 0 and epoch == start_epoch:
                print(f"    First batch on GPU: {patches_gpu.shape}, starting forward...", flush=True)
            label_t = torch.tensor(label, dtype=torch.long, device=device)

            with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
                embeddings = chunked_encoder_forward(encoder, patches_gpu, chunk_size)
                logits = head(embeddings, coords_gpu, attn_temp)
                loss = criterion(logits, label_t) / accum_steps

            if use_bf16:
                loss.backward()
            else:
                scaler.scale(loss).backward()

            epoch_loss += loss.item() * accum_steps
            n_batches += 1

            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == n_train:
                if use_bf16:
                    torch.nn.utils.clip_grad_norm_(lora_params + head_params, cfg['max_grad_norm'])
                    optimizer.step()
                else:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(lora_params + head_params, cfg['max_grad_norm'])
                    scaler.step(optimizer)
                    scaler.update()
                optimizer.zero_grad(set_to_none=True)

            del patches_gpu, coords_gpu, embeddings, logits, loss

            # Print every 100 batches (was 500) AND at least every 30 seconds
            # of wall-clock time. Keeps Colab's frontend alive on slow batches.
            now = time.time()
            if (batch_idx + 1) % 100 == 0 or (now - _last_heartbeat) > 30:
                _last_heartbeat = now
                cur_loss = epoch_loss / max(n_batches, 1)
                print(f"    train [{batch_idx+1}/{n_train}] loss={cur_loss:.4f}", flush=True)

        scheduler.step()

        # ── Validate (full patch set, no augmentation) ──
        encoder.eval(); head.eval()
        preds, labels_list = [], []

        with torch.no_grad():
            for patches, coords, label, sid in val_loader:
                patches_gpu = prepare_patches_for_encoder(patches, augment=False, device=device)
                coords_gpu = coords.to(device, non_blocking=True)

                with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
                    embeddings = chunked_encoder_forward(encoder, patches_gpu, chunk_size)
                    logits = head(embeddings, coords_gpu, attn_temp)

                preds.append(logits.argmax().cpu().item())
                labels_list.append(label)
                del patches_gpu, coords_gpu, embeddings, logits

        qwk = cohen_kappa_score(labels_list, preds, weights='quadratic')
        should_stop = early_stop(qwk)

        # SPEED: only snapshot LoRA + head (~few MB), not the full 1.2GB encoder
        if qwk > best_qwk:
            best_qwk = qwk
            best_epoch = epoch + 1
            best_lora_state = _extract_trainable_state(encoder)
            best_head_state = {k: v.cpu().clone() for k, v in head.state_dict().items()}

        # SPEED: save epoch checkpoint to LOCAL SSD only (Drive sync deferred)
        cur_lora_state = _extract_trainable_state(encoder)
        cur_head_state = {k: v.cpu().clone() for k, v in head.state_dict().items()}
        ckpt_data = {
            'lora_state': cur_lora_state,
            'head_state': cur_head_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_lora_state': best_lora_state,
            'best_head_state': best_head_state,
        }
        if not use_bf16:
            ckpt_data['scaler_state_dict'] = scaler.state_dict()
        tmp_path = ckpt_path + '.tmp'
        torch.save(ckpt_data, tmp_path)
        os.replace(tmp_path, ckpt_path)

        meta = {
            'epoch': epoch,
            'best_qwk': best_qwk,
            'best_epoch': best_epoch,
            'es_counter': early_stop.counter,
            'es_best': early_stop.best,
            'train_loss': epoch_loss / n_batches,
            'val_qwk': qwk,
        }
        with open(meta_path, 'w') as f:
            json.dump(_to_jsonable(meta), f, indent=2)

        # v5: mirror to Drive after EVERY epoch (was every 3).
        # A mid-fold disconnect now costs at most ~20 min, not 60 min.
        # Drive write is small (~3MB LoRA+head state).
        if True or should_stop:  # always save
            try:
                shutil.copy2(ckpt_path, drive_resume_path)
                shutil.copy2(meta_path, drive_meta_path)
            except Exception as e:
                print(f"    [warn] Drive resume sync failed: {e}")

        epoch_time = time.time() - epoch_start
        epoch_bar.set_postfix(
            loss=f"{epoch_loss/n_batches:.3f}",
            qwk=f"{qwk:.4f}",
            best=f"{best_qwk:.4f}@{best_epoch}",
            es=f"{early_stop.counter}/{cfg['patience']}",
            t=f"{epoch_time:.0f}s"
        )

        if should_stop:
            epoch_bar.set_postfix(
                loss=f"{epoch_loss/n_batches:.3f}",
                qwk=f"{qwk:.4f}",
                best=f"{best_qwk:.4f}@{best_epoch}",
                stop="early"
            )
            break

    epoch_bar.close()

    # Restore best
    _load_trainable_state(encoder, best_lora_state)
    head.load_state_dict(best_head_state)
    encoder.to(device).eval()
    head.to(device).eval()

    # Final evaluation
    final_preds, final_labels, final_sids = [], [], []
    with torch.no_grad():
        final_prefetcher = CUDAPrefetcher(val_loader, device, augment=False)
        for patches_gpu, coords_gpu, label, sid in final_prefetcher:
            with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
                embeddings = chunked_encoder_forward(encoder, patches_gpu, chunk_size)
                logits = head(embeddings, coords_gpu, attn_temp)
            final_preds.append(logits.argmax().cpu().item())
            final_labels.append(label)
            final_sids.append(sid)
            del patches_gpu, coords_gpu, embeddings, logits

    metrics = compute_detailed_metrics(final_labels, final_preds)
    metrics['best_epoch'] = best_epoch
    metrics['predictions'] = final_preds
    metrics['labels'] = final_labels
    metrics['slide_ids'] = final_sids

    # Cleanup local epoch ckpt + drive resume copy (best is saved by run_stage_2)
    for path in [ckpt_path, meta_path, ckpt_path + '.tmp',
                 drive_resume_path, drive_meta_path]:
        if os.path.exists(path):
            try:
                os.remove(path)
            except OSError:
                pass

    # NOTE: we now return ONLY the LoRA adapter state, not the full encoder.
    # run_stage_2 saves these compact states to Drive as the per-fold checkpoint.
    return metrics, best_lora_state, best_head_state

print("LoRA training loop defined.")


LoRA training loop defined.


## ♻️ Recovering from a runtime disconnect

If you lost a long-running session (e.g. Colab's 24-hour wall):

1. **Stage 0 and Stage 1 outputs are safe on Drive.** Re-running this notebook will skip both stages instantly — they detect existing files in `PANDA_LoRA/patch_indices/` and `PANDA_LoRA/extracted_patches/`.
2. **The local SSD is wiped.** Set `SKIP_SSD_COPY = True` (default) to stream the 175 GB of patches directly from Drive into the 167 GB RAM cache during Stage 2 preload. This avoids the multi-hour Drive→SSD copy entirely.
3. **Per-fold checkpoints survive on Drive** (mirrored every 3 epochs to `PANDA_LoRA/checkpoints/lora_fold{N}_resume.pt`). Any completed folds skip; in-progress folds resume from epoch 3, 6, or 9.
4. Reconnect on the same A100 + High-RAM runtime and run all cells top-to-bottom. The pipeline picks up where it left off.

## ⚡ v4 changes (read before running)

**Why the last run froze at fold 0 epoch 1:** Colab's frontend disconnects an output stream that goes silent for too long. With prints every 500 batches and 38% of slides lazy-loading from Drive, gaps between prints stretched past Colab's tolerance and the cell's stdout froze (the GPU was probably still training underneath).

**What this version changes:**
- **Threaded preload** (8 workers) — Drive reads are I/O-bound, so the preload now runs ~3-4× faster (~45 min instead of ~3.7 hours)
- **Larger RAM cache** — 145 GB target instead of 108 GB, fits ~85% of slides instead of 62%
- **Training heartbeat** — prints every 100 batches *and* at least every 30 seconds of wall-clock time, with `flush=True`. Frontend connection stays alive even when batches stall on Drive reads.

**Keep your browser tab in the foreground.** Colab still expects a foreground tab. If you're stepping away for hours, paste this in the browser console (F12) before leaving:
```javascript
setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
```

**Resume:** Stages 0/1 skip. RAM preload runs again (~45 min). Stage 2 fold 0 starts fresh — there's no per-fold checkpoint yet because the previous run didn't reach epoch 3 (the first save point). Subsequent folds resume normally.

## ⚡ v5: single-fold-per-session workflow (READ FIRST)

Previous versions kept hitting Colab's 24h wall mid-training. v5 splits Stage 2 across **5 short sessions** instead of one long one. Each session takes ~3.5 hours — well below any disconnect risk.

**Each session, do exactly this:**
1. In the config cell, set `FOLD_TO_RUN` to the fold you want to train this session (`0`, `1`, `2`, `3`, or `4`).
2. Verify the HF token is still in cell 8.
3. Paste this in the browser console (F12) before starting: `setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);`
4. Run all cells top-to-bottom. Stages 0/1 skip instantly. RAM preload runs ~45 min. Fold trains ~2.5 hours. `lora_fold{N}_best.pt` saves to Drive.
5. Close the tab. Rest. Tomorrow: change `FOLD_TO_RUN` to the next number, repeat.

**Within a fold, every epoch is checkpointed to Drive.** If the runtime dies at epoch 5 of 10, you resume from epoch 6 next session — you lose at most ~20 minutes of compute, not the whole fold.

**Total: 5 sessions × ~3.5h = ~17.5h of compute, spread across 5 days.**

In [15]:
def run_stage_2():
    """Stage 2: Run 5-fold LoRA training."""
    print("=" * 60)
    print("STAGE 2: LoRA Training (5-Fold CV)")
    print("=" * 60)

    # Load labels
    df = load_labels()
    all_slide_ids = df['image_id'].tolist()
    print(f"Total slides: {len(df)}")

    # Check which slides have extracted patches
    available = {f.replace('.pt', '') for f in os.listdir(EXTRACTED_PATCHES_DIR)
                 if f.endswith('.pt')}
    df = df[df['image_id'].isin(available)].reset_index(drop=True)
    print(f"Slides with patches: {len(df)}")

    # Class weights
    class_counts = df['isup_grade'].value_counts().sort_index()
    total = len(df)
    weights = total / (len(class_counts) * class_counts.values)
    class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
    print(f"Class distribution:\n{class_counts.to_dict()}")
    print(f"Class weights: {[f'{w:.2f}' for w in weights]}")

    # ═══ DATA SOURCE: SSD or Drive ═══
    if SKIP_SSD_COPY:
        # Stream patches directly from Drive into the global RAM cache.
        # Saves the multi-hour SSD copy step. RAM cache + lazy Drive
        # fallback handles I/O during training.
        patches_dir = EXTRACTED_PATCHES_DIR  # read directly from Drive
        src_files = {f for f in os.listdir(EXTRACTED_PATCHES_DIR) if f.endswith('.pt')}
        final_count = len(src_files)
        print(f"SKIP_SSD_COPY=True: streaming from Drive ({final_count} files), "
              f"skipping the multi-hour SSD copy")
    else:
        # ═══ ROBUST DRIVE → LOCAL SSD COPY ═══
        # Handles Drive disconnects, throttling, and partial copies.
        # Resumes from where it left off if interrupted.
        os.makedirs(LOCAL_PATCHES_DIR, exist_ok=True)
        src_files = {f for f in os.listdir(EXTRACTED_PATCHES_DIR) if f.endswith('.pt')}
        local_files = {f for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt')}

        # Remove corrupt files (0-byte from failed copies)
        corrupt = set()
        for f in local_files:
            fpath = os.path.join(LOCAL_PATCHES_DIR, f)
            try:
                if os.path.getsize(fpath) < 1000:
                    corrupt.add(f)
                    os.remove(fpath)
            except OSError:
                corrupt.add(f)
        if corrupt:
            print(f"Removed {len(corrupt)} corrupt files")
            local_files -= corrupt

        to_copy = sorted(src_files - local_files)
        print(f"Local SSD: {len(local_files)}/{len(src_files)} files")

        if to_copy:
            # PRE-FLIGHT: check disk space before starting
            import shutil as _shutil_check
            free_gb = _shutil_check.disk_usage(LOCAL_PATCHES_DIR).free / 1e9
            # Estimate need: avg file size × number of files to copy
            if local_files:
                sample_size = sum(os.path.getsize(os.path.join(LOCAL_PATCHES_DIR, f))
                                  for f in list(local_files)[:5]) / max(1, min(5, len(local_files)))
            else:
                # Probe one file from source
                probe_files = list(src_files)[:1]
                sample_size = os.path.getsize(os.path.join(EXTRACTED_PATCHES_DIR, probe_files[0])) if probe_files else 19_000_000
            need_gb = (sample_size * len(to_copy)) / 1e9
            print(f"Disk: {free_gb:.1f} GB free, need ~{need_gb:.1f} GB for {len(to_copy)} files")
            if need_gb > free_gb - 5:  # 5 GB safety margin
                raise RuntimeError(
                    f"Insufficient disk space: need {need_gb:.1f} GB but only {free_gb:.1f} GB free.\n"
                    f"Free up space or use a runtime with more disk."
                )
            print(f"Copying {len(to_copy)} files to local SSD (THROTTLE-RESISTANT mode)...")
            # Drive throttling avoidance:
            #   - Smaller waves (50 vs 200), fewer workers (2 vs 4)
            #   - Longer pauses between waves
            #   - Adaptive backoff if a wave fails too much
            #   - Sequential probe first to confirm Drive is healthy
            # ULTRA-CONSERVATIVE: previous attempts caused OOM kills (load avg 996, runtime disconnect)
            # Sequential copy is slow but cannot OOM the runtime.
            MAX_RETRIES = 4
            WAVE_SIZE = 20               # tiny waves
            N_WORKERS = 1                # SEQUENTIAL — no parallelism = no thread blowup
            WAVE_PAUSE_SEC = 5           # short pause between waves
            FAIL_BACKOFF_SEC = 180       # 3 min if a wave fails >50%
            PROBE_FILES = 3
            RAM_ABORT_GB = 150           # abort if system RAM usage exceeds this (167 total)

            def _copy_one_robust(fname):
                src_path = os.path.join(EXTRACTED_PATCHES_DIR, fname)
                dst_path = os.path.join(LOCAL_PATCHES_DIR, fname)
                tmp_path = dst_path + '.tmp'
                last_err = None
                for attempt in range(MAX_RETRIES):
                    try:
                        # Diagnose BEFORE copying — gives clearer error messages
                        if not os.path.exists(src_path):
                            raise FileNotFoundError(f"src missing on Drive: {fname}")
                        src_size = os.path.getsize(src_path)
                        if src_size < 1000:
                            raise OSError(f"src too small ({src_size}B): {fname}")
                        shutil.copy2(src_path, tmp_path)
                        dst_size = os.path.getsize(tmp_path)
                        if dst_size != src_size:
                            os.remove(tmp_path)
                            raise OSError(f"size mismatch: src={src_size}B dst={dst_size}B")
                        os.replace(tmp_path, dst_path)
                        return fname, True, None
                    except Exception as e:
                        last_err = f"{type(e).__name__}: {e}"
                        if os.path.exists(tmp_path):
                            try: os.remove(tmp_path)
                            except: pass
                        # Don't retry on non-recoverable errors
                        if isinstance(e, FileNotFoundError):
                            return fname, False, last_err
                        time.sleep(min(2 ** attempt, 60))
                return fname, False, last_err or f"failed {MAX_RETRIES} retries"

            # ── PROBE: sequential test of first few files to verify Drive is responsive ──
            if len(to_copy) > PROBE_FILES:
                print(f"  Probing Drive with {PROBE_FILES} sequential files...")
                probe_t0 = time.time()
                probe_results = []  # FIX: track which probe files succeeded
                for fname in to_copy[:PROBE_FILES]:
                    _, ok, _ = _copy_one_robust(fname)
                    probe_results.append((fname, ok))
                probe_ok = sum(1 for _, ok in probe_results if ok)
                probe_failed = [fname for fname, ok in probe_results if not ok]
                probe_elapsed = time.time() - probe_t0
                avg_per_file = probe_elapsed / PROBE_FILES
                print(f"  Probe: {probe_ok}/{PROBE_FILES} OK, avg {avg_per_file:.1f}s/file")
                if probe_failed:
                    print(f"  Probe failures will be retried in main loop: {len(probe_failed)} files")

                if probe_ok < PROBE_FILES:
                    print("  WARNING: probe shows Drive failures — throttle may still be active.")
                    print(f"  Sleeping 10 minutes before retry. If you see this twice, STOP and wait several hours.")
                    time.sleep(600)
                elif avg_per_file > 5:
                    print(f"  WARNING: probe shows slow Drive ({avg_per_file:.1f}s/file > 5s).")
                    print("  Drive may be partially throttled. Continuing with extra-conservative settings.")
                    WAVE_SIZE = 25
                    N_WORKERS = 1
                    WAVE_PAUSE_SEC = 30

                # FIX: only skip probe SUCCESSES; KEEP probe failures so main loop retries them.
                to_copy = probe_failed + to_copy[PROBE_FILES:]
                local_now = {f for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt')}
                to_copy = sorted(set(to_copy) - local_now)

            # RAM monitoring helper
            def _check_ram():
                try:
                    with open('/proc/meminfo') as mf:
                        info = {}
                        for line in mf:
                            k, v = line.split(':', 1)
                            info[k.strip()] = int(v.strip().split()[0])  # kB
                    used_gb = (info['MemTotal'] - info['MemAvailable']) / 1e6
                    return used_gb
                except Exception:
                    return 0.0

            total_ok, total_fail = 0, 0
            consecutive_bad_waves = 0
            import gc
            for wave_start in range(0, len(to_copy), WAVE_SIZE):
                # Pre-wave RAM check — bail BEFORE OOM kill
                ram_gb = _check_ram()
                if ram_gb > RAM_ABORT_GB:
                    print(f"    ✗ RAM at {ram_gb:.1f}/167 GB — aborting before OOM. ")
                    print(f"    Restart runtime, then re-run this cell. Already-copied files will be skipped.")
                    break
                # Free Python-side memory between waves
                gc.collect()
                wave = to_copy[wave_start:wave_start + WAVE_SIZE]
                wave_num = wave_start // WAVE_SIZE + 1
                total_waves = (len(to_copy) + WAVE_SIZE - 1) // WAVE_SIZE

                wave_ok, wave_fail = 0, 0
                wave_errors = []  # collect first few error messages for diagnosis
                with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
                    futures = {ex.submit(_copy_one_robust, f): f for f in wave}
                    for fut in tqdm(concurrent.futures.as_completed(futures),
                                    total=len(futures),
                                    desc=f"  Wave {wave_num}/{total_waves}", leave=False):
                        _, success, err = fut.result()
                        if success:
                            wave_ok += 1
                        else:
                            wave_fail += 1
                            if len(wave_errors) < 3 and err:
                                wave_errors.append(err)

                total_ok += wave_ok
                total_fail += wave_fail
                done_total = len({f for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt')})
                fail_rate = wave_fail / max(1, len(wave))
                status_emoji = "✓" if fail_rate < 0.1 else ("⚠" if fail_rate < 0.5 else "✗")
                ram_now = _check_ram()
                print(f"    {status_emoji} Wave {wave_num}/{total_waves}: {wave_ok} OK, {wave_fail} fail | "
                      f"SSD: {done_total}/{len(src_files)} | RAM: {ram_now:.0f}GB")
                # SHOW WHY files failed — this is the missing piece
                if wave_errors:
                    for err in wave_errors:
                        print(f"      reason: {err}")

                # ── Throttle detection: if a wave fails >50%, back off hard ──
                if fail_rate > 0.5:
                    consecutive_bad_waves += 1
                    backoff = FAIL_BACKOFF_SEC * consecutive_bad_waves
                    print(f"    ⚠ Wave failure rate {fail_rate:.0%} — likely Drive throttle.")
                    print(f"    Sleeping {backoff}s ({backoff/60:.1f} min) to let Drive recover...")
                    if consecutive_bad_waves >= 3:
                        print(f"    ✗ 3 bad waves in a row. ABORTING — Drive throttle is severe.")
                        print(f"    Wait 12-24 hours, then re-run this cell. Already-copied files will be skipped.")
                        break
                    time.sleep(backoff)
                else:
                    consecutive_bad_waves = 0
                    if wave_start + WAVE_SIZE < len(to_copy):
                        time.sleep(WAVE_PAUSE_SEC)

            # Final retry for whatever's still missing (single-file sequential, slow but reliable)
            still_missing = sorted({f for f in os.listdir(EXTRACTED_PATCHES_DIR) if f.endswith('.pt')}
                                   - {f for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt')})
            if still_missing and len(still_missing) < 500:
                print(f"  Final sequential retry for {len(still_missing)} files...")
                time.sleep(30)
                retry_ok = 0
                for fname in tqdm(still_missing, desc="  Retry", leave=False):
                    _, ok, _ = _copy_one_robust(fname)
                    if ok:
                        retry_ok += 1
                    time.sleep(0.5)  # gentle pacing
                print(f"  Retry recovered: {retry_ok}/{len(still_missing)}")
            elif still_missing:
                print(f"  ⚠ {len(still_missing)} files still missing — too many for sequential retry.")
                print(f"  Re-run this cell after Drive throttle clears (12-24 hours).")

            final_count = len({f for f in os.listdir(LOCAL_PATCHES_DIR) if f.endswith('.pt')})
            print(f"SSD copy done: {final_count}/{len(src_files)} files")
            if final_count < len(src_files):
                missing = len(src_files) - final_count
                pct = 100 * final_count / len(src_files)
                print(f"  ⚠ {missing} files missing ({pct:.1f}% complete).")
                print(f"  Stage 2 training will SKIP slides without local copies.")
                if pct < 95:
                    print(f"  ✗ Less than 95% copied — recommend stopping and re-running after Drive recovers.")
                    raise RuntimeError(f"Insufficient SSD copy: only {pct:.1f}% complete. Wait and retry.")
        else:
            final_count = len(local_files)

        # Primary: SSD, Fallback: Drive
        patches_dir = LOCAL_PATCHES_DIR
        print(f"Data: local SSD ({final_count} files) + Drive fallback")

    # ═══ GLOBAL RAM PRELOAD from SSD (ONCE for all 5 folds) ═══
    # MEMORY-SAFE: leave 60+ GB headroom for training (model, optimizer,
    # activations, DataLoader workers). Previous runs OOM-crashed with only 12 GB free.
    # Lazy-loading uncached slides from local SSD is fast (~500 MB/s).
    import psutil
    total_gb = psutil.virtual_memory().total / 1e9
    available_gb = psutil.virtual_memory().available / 1e9
    # Reserve 35 GB for training (model + optimizer + activations + workers).
    # On 179 GB system → ~145 GB cache → fits ~7,800 of 9,128 slides (~85%).
    # Lazy-load for the remaining ~1,300 is fast since the global cache covers
    # the working set across all 5 folds.
    max_cache_gb = min(available_gb - 35, total_gb * 0.82, 150)
    max_cache_gb = max(max_cache_gb, 30)
    print(f"RAM total: {total_gb:.0f} GB | available: {available_gb:.0f} GB")
    print(f"Cache target: {max_cache_gb:.0f} GB (leaving {available_gb - max_cache_gb:.0f} GB free for training)")
    # v6: SKIP the global preload. It's been hanging on Drive throttling.
    # Linux's OS-level file cache will populate naturally during training:
    # first epoch reads from Drive (~45-90 min), subsequent epochs hit OS
    # cache (~10-20 min each). No Python-side caching needed.
    print("v6: Skipping global RAM preload — OS file cache will warm up during epoch 1")
    # preload_global_cache(all_slide_ids, patches_dir,
    #                      fallback_dir=EXTRACTED_PATCHES_DIR,
    #                      max_ram_gb=max_cache_gb)

    # Check for completed folds
    results_path = os.path.join(LORA_RESULTS_DIR, 'lora_training_results.json')
    if os.path.exists(results_path):
        with open(results_path, 'r') as f:
            all_results = json.load(f)
    else:
        all_results = {'fold_metrics': [], 'completed_folds': []}

    completed_folds = set(all_results.get('completed_folds', []))
    remaining_folds = [f for f in range(TRAIN_CONFIG['n_folds']) if f not in completed_folds]
    # ── FOLD_TO_RUN filter: single-fold-per-session mode ──
    if FOLD_TO_RUN is not None:
        if FOLD_TO_RUN in completed_folds:
            next_fold = sorted(set(range(5)) - completed_folds)
            next_str = next_fold[0] if next_fold else 'all done'
            print(f"⚠ Fold {FOLD_TO_RUN} already complete. Set FOLD_TO_RUN = {next_str}.")
            return all_results
        if FOLD_TO_RUN not in remaining_folds:
            print(f"⚠ Fold {FOLD_TO_RUN} not in remaining {remaining_folds}.")
            return all_results
        remaining_folds = [FOLD_TO_RUN]
        print(f"🎯 Single-fold mode: running ONLY fold {FOLD_TO_RUN}")

    if not remaining_folds:
        print("All folds complete!")
        return all_results

    print(f"Completed folds: {sorted(completed_folds)}")
    print(f"Remaining folds: {remaining_folds}")

    # Create encoder (shared across folds — LoRA state reset per fold)
    encoder = create_uni_lora_encoder()
    encoder = encoder.to(device)
    initial_encoder_state = {k: v.cpu().clone() for k, v in encoder.state_dict().items()}

    # Track best fold for later stages
    best_fold_qwk = -1
    best_fold_encoder_state = None

    for fold in remaining_folds:
        print(f"\n{'─' * 40}")
        print(f"Fold {fold}/{TRAIN_CONFIG['n_folds']-1}")
        print(f"{'─' * 40}")

        # Reset encoder to initial LoRA state
        encoder.load_state_dict(initial_encoder_state)
        encoder = encoder.to(device)

        # Fresh aggregation head
        head = create_aggregation_head().to(device)

        # Create data loaders
        train_df = df[df['kfold'] != fold].reset_index(drop=True)
        val_df = df[df['kfold'] == fold].reset_index(drop=True)

        # Both use global RAM cache (loaded ONCE above — no per-fold cost)
        train_dataset = LoRAPatchDataset(
            train_df['image_id'], train_df['isup_grade'],
            patches_dir, subsample_k=TRAIN_CONFIG['K'],
            fallback_dir=EXTRACTED_PATCHES_DIR)
        val_dataset = LoRAPatchDataset(
            val_df['image_id'], val_df['isup_grade'],
            patches_dir, subsample_k=None,
            fallback_dir=EXTRACTED_PATCHES_DIR)

        # Most slides in global RAM cache — few need SSD
        n_cached = sum(1 for s in train_df['image_id'] if s in _GLOBAL_PATCH_CACHE)
        n_uncached = len(train_df) - n_cached
        train_workers = 2 if n_uncached > 50 else 0
        val_cached = sum(1 for s in val_df['image_id'] if s in _GLOBAL_PATCH_CACHE)
        print(f"  Cache: train {n_cached}/{len(train_df)} | val {val_cached}/{len(val_df)}")

        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                                  collate_fn=lora_collate_fn, num_workers=train_workers,
                                  pin_memory=True,
                                  persistent_workers=train_workers > 0,
                                  prefetch_factor=4 if train_workers > 0 else None)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                                collate_fn=lora_collate_fn, num_workers=0)  # in RAM

        print(f"  Train: {len(train_df)} slides | Val: {len(val_df)} slides")

        # Resource utilization
        import psutil
        ram_used = psutil.virtual_memory().used / 1e9
        ram_total = psutil.virtual_memory().total / 1e9
        if torch.cuda.is_available():
            gpu_alloc = torch.cuda.memory_allocated() / 1e9
            gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"  Resources: RAM {ram_used:.1f}/{ram_total:.0f} GB | GPU {gpu_alloc:.1f}/{gpu_total:.0f} GB")

        fold_start = time.time()
        metrics, best_enc_state, best_head_state = train_one_fold_lora(
            encoder, head, fold, train_loader, val_loader,
            class_weights_tensor, device)
        fold_time = (time.time() - fold_start) / 60

        print(f"  Fold {fold} done: QWK={metrics['qwk']:.4f} | "
              f"Acc={metrics['grading_accuracy']:.4f} | "
              f"{fold_time:.1f}min | ep{metrics['best_epoch']}")

        # Save fold checkpoint (LoRA adapter weights + head only — ~few MB, not 1.2GB)
        fold_ckpt_path = os.path.join(LORA_CHECKPOINTS_DIR, f'lora_fold{fold}_best.pt')
        torch.save({
            'lora_state': best_enc_state,        # LoRA adapter weights only
            'head_state_dict': best_head_state,
            'encoder_state_dict': best_enc_state,  # back-compat alias for Stage 3
            'fold': fold,
            'qwk': metrics['qwk'],
            'best_epoch': metrics['best_epoch'],
            'lora_config': LORA_CONFIG,
            'train_config': TRAIN_CONFIG,
            'lora_only': True,
        }, fold_ckpt_path)
        print(f"  Saved: {fold_ckpt_path}")

        # Track best fold
        if metrics['qwk'] > best_fold_qwk:
            best_fold_qwk = metrics['qwk']
            best_fold_encoder_state = best_enc_state  # LoRA-only state

        # Save fold predictions
        pred_path = os.path.join(LORA_RESULTS_DIR, f'lora_fold{fold}_predictions.json')
        with open(pred_path, 'w') as f:
            json.dump(_to_jsonable({
                'slide_ids': metrics.get('slide_ids', []),
                'predictions': metrics.get('predictions', []),
                'labels': metrics.get('labels', []),
            }), f)

        # Update progress
        save_metrics = {k: v for k, v in metrics.items()
                        if k not in ['predictions', 'labels', 'slide_ids']}
        save_metrics['fold_time_min'] = fold_time
        save_metrics['fold'] = fold  # Store actual fold number
        all_results['fold_metrics'].append(save_metrics)
        all_results['completed_folds'].append(fold)

        with open(results_path, 'w') as f:
            json.dump(_to_jsonable(all_results), f, indent=2)

        del head, train_loader, val_loader
        torch.cuda.empty_cache()

        # SPEED: VRAM monitoring
        if torch.cuda.is_available():
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            print(f"  VRAM peak: {peak_gb:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
            torch.cuda.reset_peak_memory_stats()

    # Aggregate results
    fold_qwks = [m['qwk'] for m in all_results['fold_metrics']]
    fold_accs = [m['grading_accuracy'] for m in all_results['fold_metrics']]
    all_results['mean_qwk'] = float(np.mean(fold_qwks))
    all_results['std_qwk'] = float(np.std(fold_qwks))
    all_results['mean_acc'] = float(np.mean(fold_accs))
    all_results['std_acc'] = float(np.std(fold_accs))
    # best_fold = actual fold number (not list index)
    best_idx = int(np.argmax(fold_qwks))
    all_results['best_fold'] = all_results['fold_metrics'][best_idx].get('fold', best_idx)

    with open(results_path, 'w') as f:
        json.dump(_to_jsonable(all_results), f, indent=2)

    print(f"\n{'=' * 60}")
    print(f"STAGE 2 COMPLETE")
    print(f"  QWK = {all_results['mean_qwk']:.4f} +/- {all_results['std_qwk']:.4f}")
    print(f"  Acc = {all_results['mean_acc']:.4f} +/- {all_results['std_acc']:.4f}")
    print(f"  Fold QWKs: {[f'{q:.4f}' for q in fold_qwks]}")
    print(f"  Best fold: {all_results['best_fold']}")
    print(f"{'=' * 60}")

    del encoder
    torch.cuda.empty_cache()
    return all_results

# Run Stage 2
lora_results = run_stage_2()


STAGE 2: LoRA Training (5-Fold CV)
  Converting kfold 1-5 -> 0-4
Total slides: 9128
Slides with patches: 9128
Class distribution:
{0: 2758, 1: 2438, 2: 1133, 3: 825, 4: 984, 5: 990}
Class weights: ['0.55', '0.62', '1.34', '1.84', '1.55', '1.54']
SKIP_SSD_COPY=True: streaming from Drive (9128 files), skipping the multi-hour SSD copy
RAM total: 57 GB | available: 51 GB
Cache target: 30 GB (leaving 21 GB free for training)
v6: Skipping global RAM preload — OS file cache will warm up during epoch 1
All folds complete!


## 7. Stage 3 -- Fast Held-Out Feature Re-Extraction (ported from v4)

**Multi-threaded held-out feature extraction.** Replaces the old single-threaded Stage 3. Five CPU workers feed the GPU concurrently and each slide is encoded by **its own held-out fold's** LoRA checkpoint (slide in fold N -> `lora_fold{N}_best.pt`). No data leakage; the encoder never saw the slide it's encoding.

**Expected time on A100:** ~5-9 h for all 9128 slides (5-10x faster than the old single-threaded Stage 3).

**Requirements before running:**
- All 5 fold checkpoints exist on Drive (`lora_fold0_best.pt` ... `lora_fold4_best.pt`).
- HF_TOKEN set as a Colab secret (key-icon panel, left sidebar).

Outputs match V20 frozen-feature format: `{slide_id}.pt` with `features` + `coords` + `fold` keys. Compatible with Stage 4 below.


In [16]:
# ═══════════════════════════════════════════════════════════════
# Stage 3 -- v4 pipeline ported into v8 (multi-threaded, held-out)
# ═══════════════════════════════════════════════════════════════
#
# This cell defines v4's full extraction pipeline (SlidePreprocessor,
# build_uni_with_peft_lora, load_lora_state_into_model, _extract_for_fold,
# extract_lora_features) and adapts it to v8's globals.
#
# v8 already defines: SLIDES_DIR, LORA_BASE_DIR, LORA_CHECKPOINTS_DIR,
# LORA_FEATURES_DIR, LABELS_PATH, LORA_CONFIG, EXTRACTION_CONFIG,
# load_labels(), device. We bridge them to v4's expected names.

import threading, queue, shutil
from PIL import Image
import cv2
import openslide
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
import timm

# v4 globals -- wired to v8's globals
LORA_CKPT_PATTERN = 'lora_fold{fold}_best.pt'

# labels_df via v8's load_labels() so TEST_MODE works (returns full 9128 in
# production, sampled subset in TEST_MODE)
labels_df = load_labels()

# v4 cfg dict -- built once from v8 globals + sane defaults
EXTRACTION_CFG_V4 = {
    'slides_dir':              SLIDES_DIR,
    'slide_ext':               EXTRACTION_CONFIG.get('slide_ext', '.tiff'),
    'output_dir':              LORA_FEATURES_DIR,
    'progress_dir':            os.path.join(LORA_BASE_DIR, 'progress'),
    'patch_size':              EXTRACTION_CONFIG['patch_size'],          # 224
    'target_magnification':    EXTRACTION_CONFIG['target_magnification'],# 20
    'encoder_batch_size':      EXTRACTION_CONFIG.get('encoder_batch_size', 256),
    'checkpoint_every':        50,
    'min_tissue_ratio':        0.5,
    'sat_thresh':              8,
    'median_blur':             7,
    'morph_close':             4,
    'white_sat_thresh':        5,
    'black_rgb_thresh':        40,
    'min_patches_per_slide':   10,
    'n_workers':               5,
    'retry_failed':            True,
    'local_temp_dir':          '/content/local_slides_temp',
}
os.makedirs(EXTRACTION_CFG_V4['output_dir'], exist_ok=True)
os.makedirs(EXTRACTION_CFG_V4['progress_dir'], exist_ok=True)

# v4's extract_lora_features() reads `cfg = EXTRACTION_CONFIG`. We patch
# that single line further down to use EXTRACTION_CFG_V4 directly -- no
# global-state shuffling, no leakage into Stage 4 below.

# ══════════════════════════════════════════════════════════════════
# Slide Preprocessor — IDENTICAL to UNI extraction
# ══════════════════════════════════════════════════════════════════

class SlidePreprocessor:
    """HSV tissue seg + magnification normalization + white/black filtering."""
    def __init__(self, patch_size=256, target_mag=20, min_tissue_ratio=0.5,
                 sat_thresh=8, median_blur=7, morph_close=4,
                 white_sat_thresh=5, black_rgb_thresh=40):
        self.patch_size = patch_size
        self.target_mag = target_mag
        self.min_tissue_ratio = min_tissue_ratio
        self.sat_thresh = sat_thresh
        self.median_blur = median_blur
        self.morph_close = morph_close
        self.white_sat_thresh = white_sat_thresh
        self.black_rgb_thresh = black_rgb_thresh

    def get_extraction_params(self, slide):
        try:
            obj_power = float(slide.properties.get('openslide.objective-power', 20))
        except (ValueError, KeyError):
            obj_power = 20
        downsample_needed = obj_power / self.target_mag
        best_level, best_diff = 0, float('inf')
        for level in range(slide.level_count):
            diff = abs(slide.level_downsamples[level] - downsample_needed)
            if diff < best_diff:
                best_diff, best_level = diff, level
        level_ds = slide.level_downsamples[best_level]
        actual_mag = obj_power / level_ds
        mag_ratio = actual_mag / self.target_mag
        read_size = int(round(self.patch_size * mag_ratio))
        step_l0 = int(round(read_size * level_ds))
        return best_level, read_size, step_l0, actual_mag, obj_power

    def segment_tissue(self, slide):
        seg_level = 0
        for level in range(slide.level_count):
            if slide.level_downsamples[level] <= 64:
                seg_level = level
        dims = slide.level_dimensions[seg_level]
        img = np.array(slide.read_region((0, 0), seg_level, dims).convert('RGB'))
        img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        img_med = cv2.medianBlur(img_hsv[:, :, 1], self.median_blur)
        _, tissue_mask = cv2.threshold(img_med, self.sat_thresh, 255, cv2.THRESH_BINARY)
        if self.morph_close > 0:
            kernel = np.ones((self.morph_close, self.morph_close), np.uint8)
            tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel)
        return tissue_mask, seg_level

    def extract_patches_and_coords(self, slide):
        level, read_size, step_l0, _, _ = self.get_extraction_params(slide)
        tissue_mask, seg_level = self.segment_tissue(slide)
        seg_ds = slide.level_downsamples[seg_level]
        mask_h, mask_w = tissue_mask.shape
        needs_resize = (read_size != self.patch_size)
        patches, coords = [], []
        w0, h0 = slide.level_dimensions[0]
        for y in range(0, h0 - step_l0 + 1, step_l0):
            for x in range(0, w0 - step_l0 + 1, step_l0):
                mx = min(int(x / seg_ds), mask_w - 1)
                my = min(int(y / seg_ds), mask_h - 1)
                mw = max(1, min(int(step_l0 / seg_ds), mask_w - mx))
                mh = max(1, min(int(step_l0 / seg_ds), mask_h - my))
                if mw <= 0 or mh <= 0:
                    continue
                tissue_ratio = np.mean(tissue_mask[my:my+mh, mx:mx+mw] > 0)
                if tissue_ratio < self.min_tissue_ratio:
                    continue
                try:
                    patch_pil = slide.read_region((x, y), level, (read_size, read_size)).convert('RGB')
                    patch_arr = np.array(patch_pil)
                    patch_hsv = cv2.cvtColor(patch_arr, cv2.COLOR_RGB2HSV)
                    if np.mean(patch_hsv[:, :, 1]) < self.white_sat_thresh:
                        continue
                    if np.all(np.mean(patch_arr, axis=(0, 1)) < self.black_rgb_thresh):
                        continue
                    if needs_resize:
                        patch_pil = patch_pil.resize((self.patch_size, self.patch_size), Image.LANCZOS)
                        patch_arr = np.array(patch_pil)
                    patches.append(patch_arr)
                    coords.append([x, y])
                except Exception:
                    continue
        return patches, np.array(coords, dtype=np.int32) if coords else np.zeros((0, 2), dtype=np.int32)


# ══════════════════════════════════════════════════════════════════
# UNI v1 + PEFT LoRA Encoder (matches training exactly)
# ══════════════════════════════════════════════════════════════════

def build_uni_with_peft_lora(lora_cfg):
    """Load UNI v1 base + wrap with PEFT LoRA. Same as the training notebook."""
    print("Loading UNI v1 (ViT-L/16)...")
    base = timm.create_model(
        "vit_large_patch16_224",
        img_size=224, patch_size=16,
        init_values=1e-5, num_classes=0, dynamic_img_size=True,
    )
    weights_path = hf_hub_download("MahmoodLab/UNI", "pytorch_model.bin")
    base.load_state_dict(
        torch.load(weights_path, map_location="cpu", weights_only=False), strict=True)
    for p in base.parameters():
        p.requires_grad = False

    peft_cfg = LoraConfig(
        r=lora_cfg['r'],
        lora_alpha=lora_cfg['lora_alpha'],
        lora_dropout=lora_cfg['lora_dropout'],
        target_modules=lora_cfg['target_modules'],
        bias=lora_cfg['bias'],
    )
    model = get_peft_model(base, peft_cfg)
    # Count params
    n_total = sum(p.numel() for p in model.parameters())
    n_lora = sum(p.numel() for n, p in model.named_parameters() if 'lora_' in n)
    print(f"UNI+PEFT-LoRA: {n_total:,} total params, {n_lora:,} LoRA params ({100*n_lora/n_total:.3f}%)")
    return model


def load_lora_state_into_model(model, ckpt_path):
    """Load the LoRA state from a Stage-2 training checkpoint.

    v3 fix: use plain `load_state_dict(strict=False)` — EXACTLY what training
    used in `_load_trainable_state(encoder, lora_state)`. The keys in
    `ckpt['lora_state']` are already `named_parameters()` keys, not PEFT's
    state-dict-with-suffixes format, so PEFT's `set_peft_model_state_dict`
    silently doesn't match them. Direct loading works because the keys are
    already in the form the model expects.
    """
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'lora_state' not in ckpt:
        raise RuntimeError(f"Checkpoint {ckpt_path} has no 'lora_state' key. Keys: {list(ckpt.keys())}")
    lora_state = ckpt['lora_state']

    # Sample BEFORE load: grab one lora_A param so we can detect that it changed
    sample_key = next(k for k in lora_state if 'lora_A' in k)
    sample_param = dict(model.named_parameters()).get(sample_key)
    if sample_param is None:
        # Key doesn't exist as a model param — show some model keys for debug
        model_lora_sample = [n for n, _ in model.named_parameters() if 'lora_' in n][:3]
        raise RuntimeError(
            f"Checkpoint key '{sample_key}' not found in model.\n"
            f"Model LoRA keys look like: {model_lora_sample}")
    before = sample_param.detach().cpu().clone()

    # Load via PLAIN state_dict — matches training-time _load_trainable_state
    incompatible = model.load_state_dict(lora_state, strict=False)
    # `incompatible.missing_keys` will be huge (all the frozen base) — that's expected
    # `incompatible.unexpected_keys` should be empty
    if incompatible.unexpected_keys:
        print(f"  WARNING: unexpected keys: {incompatible.unexpected_keys[:3]}...")

    # Verify: the sample param should now equal the checkpoint value
    after = sample_param.detach().cpu()
    ckpt_t = lora_state[sample_key]
    changed = not torch.allclose(before, after, atol=1e-10)
    matches_ckpt = torch.allclose(after, ckpt_t, atol=1e-6)

    n_ckpt = len(lora_state)
    n_model = len([n for n, _ in model.named_parameters() if 'lora_' in n])
    qwk = ckpt.get('qwk', None)
    fold = ckpt.get('fold', None)
    print(f"  Checkpoint LoRA tensors: {n_ckpt}")
    print(f"  Model LoRA params:       {n_model}")
    print(f"  Sample key '{sample_key.split('.')[-2:]}': changed={changed}, matches_ckpt={matches_ckpt}")
    print(f"  Checkpoint metadata:     fold={fold}, QWK={qwk:.4f}" if qwk else f"  Metadata: fold={fold}")
    if not matches_ckpt:
        raise RuntimeError(
            f"LoRA load FAILED: model param doesn't match checkpoint after load. "
            f"DO NOT proceed — features would be garbage.")
    return n_ckpt


# ══════════════════════════════════════════════════════════════════
# Multi-worker extraction — one fold at a time
# ══════════════════════════════════════════════════════════════════

def _extract_for_fold(fold, slide_ids, encoder, preprocessor, cfg, completed_set, failed_set, _lock):
    N_WORKERS = cfg['n_workers']
    LOCAL_TEMP = cfg['local_temp_dir']
    os.makedirs(LOCAL_TEMP, exist_ok=True)
    batch_size = cfg['encoder_batch_size']

    _gpu_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    _gpu_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    slide_ids_q = queue.Queue()
    for sid in slide_ids:
        slide_ids_q.put(sid)
    gpu_q = queue.Queue(maxsize=N_WORKERS)
    preprocess_errors = []

    def _cpu_worker():
        while True:
            try:
                slide_id = slide_ids_q.get_nowait()
            except Exception:
                return
            src_path = os.path.join(cfg['slides_dir'], f"{slide_id}{cfg['slide_ext']}")
            local_path = os.path.join(LOCAL_TEMP, f"{slide_id}{cfg['slide_ext']}")
            used_local = False
            try:
                # Drive open with retry-via-copy fallback
                slide = None
                for attempt in range(3):
                    try:
                        if attempt == 0:
                            slide = openslide.OpenSlide(src_path)
                        else:
                            # Copy to local SSD first (more reliable than direct Drive read)
                            if not os.path.exists(local_path):
                                shutil.copy2(src_path, local_path)
                                used_local = True
                            slide = openslide.OpenSlide(local_path)
                        break
                    except Exception as _e:
                        if attempt == 2:
                            raise
                        time.sleep(2)
                patches, coords = preprocessor.extract_patches_and_coords(slide)
                slide.close()
                if len(patches) < cfg['min_patches_per_slide']:
                    with _lock:
                        preprocess_errors.append((slide_id, f"only {len(patches)} patches"))
                    gpu_q.put((slide_id, None, None, local_path if used_local else None))
                    continue
                patch_tensors = torch.from_numpy(np.stack(patches))
                coords_tensor = torch.tensor(coords, dtype=torch.float32)
                del patches
                gpu_q.put((slide_id, patch_tensors, coords_tensor, local_path if used_local else None))
            except Exception as e:
                with _lock:
                    preprocess_errors.append((slide_id, str(e)))
                gpu_q.put((slide_id, None, None, local_path if used_local and os.path.exists(local_path) else None))

    actual_workers = min(N_WORKERS, len(slide_ids))
    cpu_threads = []
    for _ in range(actual_workers):
        t = threading.Thread(target=_cpu_worker, daemon=True)
        t.start()
        cpu_threads.append(t)

    fold_completed, fold_failed = [], []
    pbar = tqdm(total=len(slide_ids), desc=f"Fold {fold}", unit="slide")
    processed = 0
    start_time = time.time()

    while processed < len(slide_ids):
        try:
            item = gpu_q.get(timeout=300)
        except Exception:
            alive = sum(1 for t in cpu_threads if t.is_alive())
            print(f"\n  Queue timeout. {alive}/{actual_workers} workers alive")
            if alive == 0:
                break
            continue
        slide_id, patch_tensors, coords_tensor, local_path = item
        processed += 1
        if patch_tensors is None:
            fold_failed.append(slide_id)
            pbar.update(1)
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass
            continue
        try:
            all_features = []
            with torch.inference_mode(), torch.amp.autocast('cuda'):
                for b in range(0, len(patch_tensors), batch_size):
                    batch_raw = patch_tensors[b:b+batch_size]
                    batch_gpu = batch_raw.to(device).permute(0, 3, 1, 2).float() / 255.0
                    # v5: no resize -- EXTRACTION_CONFIG['patch_size']=224 already matches UNI input.
                    batch_gpu = (batch_gpu - _gpu_mean) / _gpu_std
                    feats = encoder(batch_gpu).float().cpu()
                    all_features.append(feats)
                    del batch_raw, batch_gpu
            features_cat = torch.cat(all_features, dim=0)
            output_path = os.path.join(cfg['output_dir'], f"{slide_id}.pt")
            torch.save({'features': features_cat, 'coords': coords_tensor, 'fold': fold}, output_path)
            fold_completed.append(slide_id)
            del patch_tensors, all_features, features_cat, coords_tensor
            elapsed = time.time() - start_time
            rate = (processed - len(fold_failed)) / max(elapsed / 60, 0.01)
            remaining = (len(slide_ids) - processed) / max(rate, 0.01)
            pbar.set_postfix(rate=f"{rate:.1f}/min", eta=f"{remaining:.0f}min")
        except Exception as e:
            print(f"\n  GPU ERROR {slide_id}: {e}")
            fold_failed.append(slide_id)
        finally:
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass
        pbar.update(1)

        if len(fold_completed) % cfg['checkpoint_every'] == 0 and len(fold_completed) > 0:
            with _lock:
                completed_set.update(fold_completed)
                failed_set.update(fold_failed)
                _save_progress(cfg, completed_set, failed_set)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pbar.close()
    for t in cpu_threads:
        t.join(timeout=10)
    return fold_completed, fold_failed, preprocess_errors


def _save_progress(cfg, completed_set, failed_set):
    progress_path = os.path.join(cfg['progress_dir'], 'lora_extraction_progress.json')
    with open(progress_path, 'w') as f:
        json.dump({
            'completed': sorted(completed_set),
            'failed': sorted(failed_set),
            'timestamp': datetime.now().isoformat(),
        }, f)


def extract_lora_features():
    cfg = EXTRACTION_CFG_V4  # v8 port: was EXTRACTION_CONFIG in v4
    progress_path = os.path.join(cfg['progress_dir'], 'lora_extraction_progress.json')
    if os.path.exists(progress_path):
        with open(progress_path, 'r') as f:
            progress = json.load(f)
        completed_set = set(progress.get('completed', []))
        failed_set = set(progress.get('failed', []))
    else:
        completed_set, failed_set = set(), set()

    if os.path.isdir(cfg['output_dir']):
        existing = {f.replace('.pt', '') for f in os.listdir(cfg['output_dir']) if f.endswith('.pt')}
        completed_set |= existing

    fold_slides = {}
    for fold in range(5):
        sub = labels_df[labels_df['kfold'] == fold]
        ids = sub['image_id'].astype(str).tolist()
        retry_failed = cfg.get('retry_failed', True)
        if retry_failed:
            todo = [sid for sid in ids if sid not in completed_set]
        else:
            todo = [sid for sid in ids if sid not in completed_set and sid not in failed_set]
        fold_slides[fold] = todo

    print("=" * 60)
    print("LoRA FEATURE EXTRACTION v2 (PEFT, held-out fold per slide)")
    print("=" * 60)
    total_todo = sum(len(v) for v in fold_slides.values())
    print(f"Already done: {len(completed_set)}")
    print(f"To process:   {total_todo}")
    print(f"Per-fold workload: {[len(fold_slides[f]) for f in range(5)]}")

    if total_todo == 0:
        print("\nAll slides already processed!")
        return

    print()
    # FINAL: skip pre-flight slide verification. Drive FUSE is flaky on direct
    # opens; the real CPU workers below handle this with a local-copy fallback.
    # We don't want a transient Drive hiccup to abort the whole extraction.
    print("Skipping pre-flight slide verification (CPU workers handle Drive flakiness).")
    print()

    encoder = build_uni_with_peft_lora(LORA_CONFIG).to(device).eval()
    preprocessor = SlidePreprocessor(
        patch_size=cfg['patch_size'],
        target_mag=cfg['target_magnification'],
        min_tissue_ratio=cfg['min_tissue_ratio'],
        sat_thresh=cfg.get('sat_thresh', 8),
        median_blur=cfg.get('median_blur', 7),
        morph_close=cfg.get('morph_close', 4),
        white_sat_thresh=cfg.get('white_sat_thresh', 5),
        black_rgb_thresh=cfg.get('black_rgb_thresh', 40),
    )

    _lock = threading.Lock()
    overall_start = time.time()
    all_preprocess_errors = []

    for fold in range(5):
        todo = fold_slides[fold]
        if not todo:
            print(f"\n--- Fold {fold}: nothing to do (already encoded) ---")
            continue
        ckpt = os.path.join(LORA_CHECKPOINTS_DIR, LORA_CKPT_PATTERN.format(fold=fold))
        if not os.path.exists(ckpt):
            print(f"\n--- Fold {fold}: SKIPPING (no checkpoint at {ckpt}) ---")
            continue
        print(f"\n--- Fold {fold}: loading {LORA_CKPT_PATTERN.format(fold=fold)} ({len(todo)} slides) ---")
        # CRITICAL: load LoRA. raises if 0 tensors match.
        load_lora_state_into_model(encoder, ckpt)
        encoder.eval()
        f_done, f_fail, f_pre_err = _extract_for_fold(
            fold, todo, encoder, preprocessor, cfg, completed_set, failed_set, _lock)
        completed_set.update(f_done)
        failed_set.update(f_fail)
        all_preprocess_errors.extend(f_pre_err)
        _save_progress(cfg, completed_set, failed_set)
        print(f"  Fold {fold} done: {len(f_done)} ok, {len(f_fail)} failed")

    elapsed = time.time() - overall_start
    print(f"\n{'=' * 60}")
    print(f"EXTRACTION COMPLETE")
    print(f"  Wall-clock: {elapsed/3600:.2f} hours")
    print(f"  Total done: {len(completed_set)}")
    print(f"  Total failed: {len(failed_set)}")
    print(f"  Output: {cfg['output_dir']}")
    if all_preprocess_errors:
        print(f"  Preprocess errors: {len(all_preprocess_errors)}")
        for sid, err in all_preprocess_errors[:5]:
            print(f"    - {sid}: {err}")
    print(f"{'=' * 60}")

    shutil.rmtree(cfg['local_temp_dir'], ignore_errors=True)
    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def verify_slide_reading(slides_dir, slide_ext, n_samples=5):
    print("=" * 60)
    print("SLIDE READING VERIFICATION")
    print("=" * 60)
    slide_files = sorted([f for f in os.listdir(slides_dir) if f.endswith(slide_ext)])
    if not slide_files:
        print(f"  ERROR: No {slide_ext} files in {slides_dir}")
        return False
    import random
    random.seed(42)
    samples = random.sample(slide_files, min(n_samples, len(slide_files)))
    test_preprocessor = SlidePreprocessor()
    issues = []
    for fname in samples:
        fpath = os.path.join(slides_dir, fname)
        slide = openslide.OpenSlide(fpath)
        level, read_size, step_l0, actual_mag, obj_power = test_preprocessor.get_extraction_params(slide)
        level_dims = slide.level_dimensions[level]
        needs_resize = (read_size != 256)
        if actual_mag >= 19.0:
            status = "OK"
        elif actual_mag >= 10.0:
            status = "WARNING"
            issues.append(f"{fname}: low mag {actual_mag:.1f}x")
        else:
            status = "FAIL"
            issues.append(f"{fname}: very low mag {actual_mag:.1f}x")
        resize_note = f" -> resize {read_size}->256" if needs_resize else ""
        print(f"  {fname}: obj={obj_power}x, level={level} ({level_dims[0]}x{level_dims[1]}), eff={actual_mag:.1f}x{resize_note} [{status}]")
        slide.close()
    return len(issues) == 0


print("Stage 3 pipeline ready (v4 multi-threaded, held-out fold per slide).")

# ═══════════════════════════════════════════════════════════════
# Stage 3 entry point (replaces v6/v8 single-threaded run_stage_3)
# ═══════════════════════════════════════════════════════════════
def run_stage_3():
    """Stage 3: multi-threaded held-out feature extraction with LoRA.

    Each slide in fold N is encoded using lora_fold{N}_best.pt -- no data
    leakage. 5 CPU workers feed the GPU concurrently for ~5-10x speedup
    over the prior single-threaded loop.
    """
    return extract_lora_features()

# Run Stage 3
run_stage_3()


  Converting kfold 1-5 -> 0-4
Stage 3 pipeline ready (v4 multi-threaded, held-out fold per slide).
LoRA FEATURE EXTRACTION v2 (PEFT, held-out fold per slide)
Already done: 9128
To process:   0
Per-fold workload: [0, 0, 0, 0, 0]

All slides already processed!


## 8. Stage 4 — V20 Pipeline Re-Run with LoRA FeaturesRun the exact V20 training pipeline (all 12 experiments) using LoRA-adapted featuresinstead of frozen features. This gives a direct comparison.

In [ ]:
def run_stage_4():
    """Stage 4: Run V20 pipeline with LoRA features.

    This is essentially the V20 training loop but pointing at LORA_FEATURES_DIR.
    We import the V20 training infrastructure and swap the features directory.
    """
    print("=" * 60)
    print("STAGE 4: V20 Re-Run with LoRA Features")
    print("=" * 60)
    print(f"Features: {LORA_FEATURES_DIR}")

    # Verify features exist (scale floor with active label set so TEST_MODE works)
    n_features = len([f for f in os.listdir(LORA_FEATURES_DIR) if f.endswith('.pt')])
    expected_n = len(load_labels())
    min_required = max(1, int(expected_n * 0.9))  # tolerate 10% extraction failures
    print(f"Available LoRA features: {n_features} (expected ~{expected_n})")
    if n_features < min_required:
        print(f"  WARNING: Only {n_features} features found, need at least {min_required}. "
              f"Run Stage 3 first!")
        return

    # ── V20 Dataset (preload LoRA features) ──
    class V20Dataset(Dataset):
        _cache = {}

        def __init__(self, slide_ids, labels, features_dir):
            self.slide_ids = list(slide_ids)
            self.labels = list(labels)
            self.features_dir = features_dir

        @classmethod
        def preload_all(cls, all_ids, features_dir, max_workers=16):  # A100: more threads
            to_load = [sid for sid in all_ids if sid not in cls._cache]
            if not to_load:
                print(f"V20 cache loaded ({len(cls._cache)} slides)")
                return
            print(f"Preloading {len(to_load)} LoRA features...")
            t0 = time.time()
            loaded = 0

            def _load(sid):
                try:
                    data = torch.load(os.path.join(features_dir, f'{sid}.pt'),
                                      map_location='cpu', weights_only=False)
                    features = data['features'].float()
                    if features.ndim == 3:
                        features = features.squeeze(0)
                    coords = data.get('coords', torch.zeros(features.shape[0], 2))
                    if isinstance(coords, np.ndarray):
                        coords = torch.tensor(coords, dtype=torch.float32)
                    coords = coords.float()
                    if coords.ndim == 3:
                        coords = coords.squeeze(0)
                    return sid, features, coords
                except:
                    return sid, None, None

            with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
                for sid, feat, coord in ex.map(_load, to_load):
                    if feat is not None:
                        cls._cache[sid] = (feat, coord)
                        loaded += 1

            mem_gb = sum(f.nelement()*f.element_size() + c.nelement()*c.element_size()
                        for f, c in cls._cache.values()) / 1e9
            print(f"Loaded {loaded} slides in {time.time()-t0:.1f}s ({mem_gb:.1f}GB)")

        def __len__(self):
            return len(self.slide_ids)

        def __getitem__(self, idx):
            sid = self.slide_ids[idx]
            feat, coord = self._cache[sid]
            return feat, coord, self.labels[idx], sid

    # ── V20 Experiments ──
    V20_EXPERIMENTS = [
        ('ABMIL_defaultHP',             'default'),
        ('ABMIL_bestHP',                'best'),
        ('DualBranch_defaultHP',        'default'),
        ('DualBranch_bestHP',           'best'),
        ('RetNet_defaultHP',            'default'),
        ('RetNet_bestHP',               'best'),
        ('SpatialMult_defaultHP',       'default'),
        ('SpatialMult_bestHP',          'best'),
        ('SpatialMultRetNet_defaultHP', 'default'),
        ('SpatialMultRetNet_bestHP',    'best'),
        ('SpatialAdd_defaultHP',        'default'),
        ('SpatialAdd_bestHP',           'best'),
    ]

    # V20 auto-tuned LR to 2e-4 (from best_lr.json)
    V20_HP_SETS = {
        'default': {
            'attention_temperature': 1.0,
            'dropout': 0.25,
            'optimizer': 'adam',
            'lr': 1e-4,
            'weight_decay': 1e-5,
            'loss': 'ce',
            'label_smoothing': 0.1,
            'focal_gamma': None,
            'gamma_values': [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.98, 0.99],
        },
        'best': {
            'attention_temperature': 0.7,
            'dropout': 0.15,
            'optimizer': 'adamw',
            'lr': 2e-4,
            'weight_decay': 1e-5,
            'loss': 'focal',
            'label_smoothing': 0.1,
            'focal_gamma': 1.0,
            'gamma_values': [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
        },
    }

    def v20_create_model(exp_name, hp_name):
        hp = V20_HP_SETS[hp_name].copy()
        d, g = hp['dropout'], hp['gamma_values']
        c = V20_CONFIG

        if exp_name.startswith('ABMIL'):
            return ABMIL(c['input_dim'], c['hidden_dim'], c['n_classes'], d), hp
        elif exp_name.startswith('DualBranch'):
            return DualBranchModel(c['input_dim'], c['hidden_dim'], c['n_classes'], d, c['K'],
                                   retention_module=None, needs_coords=False), hp
        elif exp_name.startswith('RetNet'):
            ret = RetentionBlock(hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
                                 use_spatial=False, use_norm=True, norm_type='group',
                                 num_groups=c['num_groups'], use_residual=True, use_gate=True,
                                 use_ffn=True, ffn_expansion=c['ffn_expansion'], dropout=d)
            return DualBranchModel(c['input_dim'], c['hidden_dim'], c['n_classes'], d, c['K'],
                                   retention_module=ret, needs_coords=False), hp
        elif exp_name.startswith('SpatialMult_') and 'RetNet' not in exp_name:
            sp = SpatialRetentionMult(hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
                                      gamma_values=g)
            return DualBranchModel(c['input_dim'], c['hidden_dim'], c['n_classes'], d, c['K'],
                                   retention_module=sp, needs_coords=True), hp
        elif exp_name.startswith('SpatialMultRetNet'):
            ret = RetentionBlock(hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
                                 gamma_values=g, use_spatial=True, spatial_mode='multiplicative',
                                 use_norm=True, norm_type='group', num_groups=c['num_groups'],
                                 use_residual=True, use_gate=True, use_ffn=True,
                                 ffn_expansion=c['ffn_expansion'], dropout=d)
            return DualBranchModel(c['input_dim'], c['hidden_dim'], c['n_classes'], d, c['K'],
                                   retention_module=ret, needs_coords=True), hp
        elif exp_name.startswith('SpatialAdd'):
            sp = SpatialRetentionAdd(hidden_dim=c['hidden_dim'], num_heads=c['num_heads'],
                                      gamma_values=g)
            return DualBranchModel(c['input_dim'], c['hidden_dim'], c['n_classes'], d, c['K'],
                                   retention_module=sp, needs_coords=True), hp
        else:
            raise ValueError(f"Unknown: {exp_name}")

    def v20_create_loss(hp, class_weights_tensor):
        w = class_weights_tensor  # V20 always uses class weights
        if hp['loss'] == 'focal':
            return FocalLoss(6, hp['focal_gamma'], hp['label_smoothing'], w)
        return LabelSmoothingLoss(6, hp['label_smoothing'], w)

    # Load data
    df = load_labels()
    available = {f.replace('.pt', '') for f in os.listdir(LORA_FEATURES_DIR) if f.endswith('.pt')}
    df = df[df['image_id'].isin(available)].reset_index(drop=True)

    class_counts = df['isup_grade'].value_counts().sort_index()
    weights = len(df) / (len(class_counts) * class_counts.values)
    class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

    V20Dataset.preload_all(df['image_id'].tolist(), LORA_FEATURES_DIR)

    # Try GPU packing
    sids = sorted(V20Dataset._cache.keys())
    total_patches = sum(V20Dataset._cache[s][0].shape[0] for s in sids)
    feat_dim = next(iter(V20Dataset._cache.values()))[0].shape[1]
    needed_gb = (total_patches * feat_dim * 2 + total_patches * 2 * 4) / 1e9
    print(f"GPU packing: {total_patches:,} patches x {feat_dim}d = {needed_gb:.1f}GB")

    gpu_packed = False
    if torch.cuda.is_available():
        free = torch.cuda.mem_get_info(device)[0] / 1e9
        if needed_gb < free * 0.85:
            print("Packing features to GPU...")
            all_feat = torch.empty(total_patches, feat_dim, dtype=torch.float16, device=device)
            all_coord = torch.empty(total_patches, 2, dtype=torch.float32, device=device)
            gpu_offsets = {}
            idx = 0
            for sid in sids:
                f, c = V20Dataset._cache[sid]
                n = f.shape[0]
                all_feat[idx:idx+n] = f.half().to(device)
                all_coord[idx:idx+n] = c.float().to(device)
                gpu_offsets[sid] = (idx, idx+n)
                idx += n
            V20Dataset._cache.clear()
            import gc; gc.collect()
            gpu_packed = True
            print(f"GPU packed: {needed_gb:.1f}GB")

    # Override __getitem__ if GPU packed
    if gpu_packed:
        _orig_getitem = V20Dataset.__getitem__
        def _gpu_getitem(self, idx):
            sid = self.slide_ids[idx]
            start, end = gpu_offsets[sid]
            return all_feat[start:end].float(), all_coord[start:end].float(), self.labels[idx], sid
        V20Dataset.__getitem__ = _gpu_getitem

    # Results tracking
    v20_results_path = os.path.join(LORA_RESULTS_DIR, 'v20_lora_features_results.json')
    if os.path.exists(v20_results_path):
        with open(v20_results_path, 'r') as f:
            v20_results = json.load(f)
    else:
        v20_results = {}

    # Fold-level progress tracking
    fold_progress_path = os.path.join(LORA_RESULTS_DIR, 'v20_fold_progress.json')
    if os.path.exists(fold_progress_path):
        with open(fold_progress_path, 'r') as f:
            fold_progress = json.load(f)
    else:
        fold_progress = {}

    # Run experiments
    for exp_idx, (exp_name, hp_name) in enumerate(V20_EXPERIMENTS):
        if exp_name in v20_results:
            print(f"  [{exp_idx+1}/{len(V20_EXPERIMENTS)}] {exp_name}: "
                  f"already done (QWK={v20_results[exp_name]['mean_qwk']:.4f})")
            continue

        print(f"\n  [{exp_idx+1}/{len(V20_EXPERIMENTS)}] {exp_name}")

        # Resume fold progress for this experiment
        exp_progress = fold_progress.get(exp_name, {'completed_folds': [], 'fold_qwks': {}})
        completed_folds = set(exp_progress['completed_folds'])
        fold_qwks_dict = exp_progress['fold_qwks']  # {fold_str: qwk}

        remaining_folds = [f for f in range(5) if f not in completed_folds]
        if completed_folds:
            print(f"    Resuming: folds {sorted(completed_folds)} done, "
                  f"{len(remaining_folds)} remaining")

        exp_start = time.time()

        for fold in remaining_folds:
            fold_start = time.time()
            train_df = df[df['kfold'] != fold].reset_index(drop=True)
            val_df = df[df['kfold'] == fold].reset_index(drop=True)

            train_ds = V20Dataset(train_df['image_id'], train_df['isup_grade'], LORA_FEATURES_DIR)
            val_ds = V20Dataset(val_df['image_id'], val_df['isup_grade'], LORA_FEATURES_DIR)
            train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                                      collate_fn=lambda b: b[0], num_workers=0)
            val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                                    collate_fn=lambda b: b[0], num_workers=0)

            model, hp = v20_create_model(exp_name, hp_name)
            model = model.to(device)
            criterion = v20_create_loss(hp, class_weights_tensor)
            attn_temp = hp['attention_temperature']

            if hp['optimizer'] == 'adamw':
                opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'],
                                        weight_decay=hp['weight_decay'])
            else:
                opt = torch.optim.Adam(model.parameters(), lr=hp['lr'],
                                       weight_decay=hp['weight_decay'])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
            es = EarlyStopping(patience=10, threshold=0.001)
            scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

            best_qwk_f = -1
            best_epoch = 0
            best_state = None

            ep_bar = tqdm(range(50), desc=f"    fold {fold}", leave=False)
            for epoch in ep_bar:
                model.train()
                for feat, coord, label, _ in train_loader:
                    feat = feat.to(device, non_blocking=True)
                    label_t = torch.tensor(label, dtype=torch.long, device=device)
                    opt.zero_grad()
                    with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                        if model.needs_coords:
                            coord = coord.to(device, non_blocking=True)
                            out = model(feat, coord, attn_temp)
                        else:
                            out = model(feat, temperature=attn_temp)
                        loss = criterion(out, label_t)
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                sched.step()

                model.eval()
                preds, labs = [], []
                with torch.no_grad():
                    for feat, coord, label, _ in val_loader:
                        feat = feat.to(device, non_blocking=True)
                        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                            if model.needs_coords:
                                coord = coord.to(device, non_blocking=True)
                                out = model(feat, coord, attn_temp)
                            else:
                                out = model(feat, temperature=attn_temp)
                        preds.append(out.argmax().cpu().item())
                        labs.append(label)

                qwk = cohen_kappa_score(labs, preds, weights='quadratic')
                if qwk > best_qwk_f:
                    best_qwk_f = qwk
                    best_epoch = epoch + 1
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                ep_bar.set_postfix(qwk=f"{qwk:.4f}", best=f"{best_qwk_f:.4f}")
                if es(qwk):
                    break
            ep_bar.close()

            # Restore best model for this fold
            if best_state is not None:
                model.load_state_dict(best_state)
            del best_state

            fold_time = (time.time() - fold_start) / 60
            print(f"    Fold {fold}: QWK={best_qwk_f:.4f} | ep{best_epoch} | {fold_time:.1f}min")

            # Save fold-level progress
            fold_qwks_dict[str(fold)] = best_qwk_f
            completed_folds.add(fold)
            exp_progress = {
                'completed_folds': sorted(completed_folds),
                'fold_qwks': fold_qwks_dict,
            }
            fold_progress[exp_name] = exp_progress
            with open(fold_progress_path, 'w') as f:
                json.dump(_to_jsonable(fold_progress), f, indent=2)

            del model, opt, sched
            torch.cuda.empty_cache()

        # Aggregate all 5 folds
        fold_qwks = [fold_qwks_dict[str(f)] for f in range(5)]
        result = {
            'fold_qwks': fold_qwks,
            'mean_qwk': float(np.mean(fold_qwks)),
            'std_qwk': float(np.std(fold_qwks)),
        }
        v20_results[exp_name] = result

        exp_time = (time.time() - exp_start) / 60
        print(f"  {exp_name}: QWK = {result['mean_qwk']:.4f} +/- {result['std_qwk']:.4f} "
              f"({exp_time:.1f}min)")

        # Save experiment result and clean fold progress
        with open(v20_results_path, 'w') as f:
            json.dump(_to_jsonable(v20_results), f, indent=2)
        if exp_name in fold_progress:
            del fold_progress[exp_name]
            with open(fold_progress_path, 'w') as f:
                json.dump(_to_jsonable(fold_progress), f, indent=2)

    # Summary comparison
    print(f"\n{'=' * 60}")
    print("STAGE 4 COMPLETE — LoRA Features vs Frozen Features")
    print(f"{'=' * 60}")
    print(f"{'Experiment':<35s} {'LoRA QWK':>10s}")
    print("-" * 50)
    for exp_name, _ in V20_EXPERIMENTS:
        if exp_name in v20_results:
            r = v20_results[exp_name]
            print(f"{exp_name:<35s} {r['mean_qwk']:.4f} +/- {r['std_qwk']:.4f}")

    # Cleanup
    V20Dataset._cache.clear()
    import gc; gc.collect()
    torch.cuda.empty_cache()

    return v20_results

# Run Stage 4
v20_lora_results = run_stage_4()

STAGE 4: V20 Re-Run with LoRA Features
Features: /content/drive/MyDrive/PANDA_LoRA/lora_features
  Converting kfold 1-5 -> 0-4
Available LoRA features: 9128 (expected ~9128)
  Converting kfold 1-5 -> 0-4
Preloading 9128 LoRA features...
Loaded 9128 slides in 2237.0s (22.6GB)
GPU packing: 5,516,732 patches x 1024d = 11.3GB
Packing features to GPU...
GPU packed: 11.3GB
  [1/12] ABMIL_defaultHP: already done (QWK=0.9317)
  [2/12] ABMIL_bestHP: already done (QWK=0.9457)
  [3/12] DualBranch_defaultHP: already done (QWK=0.9362)
  [4/12] DualBranch_bestHP: already done (QWK=0.9464)
  [5/12] RetNet_defaultHP: already done (QWK=0.9411)
  [6/12] RetNet_bestHP: already done (QWK=0.9503)
  [7/12] SpatialMult_defaultHP: already done (QWK=0.9516)
  [8/12] SpatialMult_bestHP: already done (QWK=0.9501)
  [9/12] SpatialMultRetNet_defaultHP: already done (QWK=0.9419)
  [10/12] SpatialMultRetNet_bestHP: already done (QWK=0.9498)
  [11/12] SpatialAdd_defaultHP: already done (QWK=0.9529)

  [12/12] Spatial

    Fold 0: QWK=0.9475 | ep4 | 28.7min


    Fold 1: QWK=0.9433 | ep10 | 28.8min


    Fold 2: QWK=0.9443 | ep7 | 35.0min


    Fold 3: QWK=0.9511 | ep5 | 30.5min


    Fold 4: QWK=0.9496 | ep3 | 26.8min
  SpatialAdd_bestHP: QWK = 0.9472 +/- 0.0030 (149.9min)

STAGE 4 COMPLETE — LoRA Features vs Frozen Features
Experiment                            LoRA QWK
--------------------------------------------------
ABMIL_defaultHP                     0.9317 +/- 0.0051
ABMIL_bestHP                        0.9457 +/- 0.0059
DualBranch_defaultHP                0.9362 +/- 0.0078
DualBranch_bestHP                   0.9464 +/- 0.0055
RetNet_defaultHP                    0.9411 +/- 0.0015
RetNet_bestHP                       0.9503 +/- 0.0041
SpatialMult_defaultHP               0.9516 +/- 0.0037
SpatialMult_bestHP                  0.9501 +/- 0.0047
SpatialMultRetNet_defaultHP         0.9419 +/- 0.0046
SpatialMultRetNet_bestHP            0.9498 +/- 0.0022
SpatialAdd_defaultHP                0.9529 +/- 0.0023
SpatialAdd_bestHP                   0.9472 +/- 0.0030


## 9. Results Summary & Comparison

In [ ]:
def print_comparison():
    """Print LoRA vs Frozen feature comparison."""

    # Load LoRA Stage 2 results (end-to-end LoRA)
    s2_path = os.path.join(LORA_RESULTS_DIR, 'lora_training_results.json')
    if os.path.exists(s2_path):
        with open(s2_path, 'r') as f:
            s2 = json.load(f)
        print("\n" + "=" * 60)
        print("END-TO-END LoRA TRAINING (Stage 2)")
        print("=" * 60)
        print(f"  LoRA UNI v1 + SpatialMultRetNet: "
              f"QWK = {s2['mean_qwk']:.4f} +/- {s2['std_qwk']:.4f}")
        print(f"  Fold QWKs: {[f'{q:.4f}' for q in [m['qwk'] for m in s2['fold_metrics']]]}")

    # Load V20 frozen baselines (from V20 results on Drive)
    print("\n" + "=" * 60)
    print("COMPARISON: LoRA vs Frozen Features")
    print("=" * 60)
    print(f"{'Method':<45s} {'QWK':>12s}")
    print("-" * 60)

    # V20 frozen baselines (hardcoded from V20 results)
    baselines = [
        ("Frozen UNI v1 + ABMIL (V20)", 0.946),
        ("Frozen UNI v1 + RetNet (V20)", 0.954),
        ("Frozen UNI v1 + SpatialMultRetNet (V20)", 0.952),
    ]
    for name, qwk in baselines:
        print(f"  {name:<43s} {qwk:.4f}")

    print()
    if os.path.exists(s2_path):
        print(f"  {'LoRA UNI v1 + SpatialMultRetNet (Stage 2)':<43s} "
              f"{s2['mean_qwk']:.4f} +/- {s2['std_qwk']:.4f}")

    # Load Stage 4 results
    s4_path = os.path.join(LORA_RESULTS_DIR, 'v20_lora_features_results.json')
    if os.path.exists(s4_path):
        with open(s4_path, 'r') as f:
            s4 = json.load(f)
        print()
        for exp_name in ['SpatialMultRetNet_bestHP', 'RetNet_bestHP', 'ABMIL_bestHP']:
            if exp_name in s4:
                r = s4[exp_name]
                print(f"  {'LoRA features + ' + exp_name:<43s} "
                      f"{r['mean_qwk']:.4f} +/- {r['std_qwk']:.4f}")

    # Verdict
    if os.path.exists(s2_path):
        lora_qwk = s2['mean_qwk']
        print(f"\n{'─' * 60}")
        if lora_qwk > 0.960:
            print("  VERDICT: CLEAR WIN — LoRA significantly beats frozen baselines")
        elif lora_qwk > 0.955:
            print("  VERDICT: MARGINAL WIN — LoRA shows modest improvement")
        elif lora_qwk > 0.952:
            print("  VERDICT: WITHIN NOISE — LoRA ≈ frozen features")
        else:
            print("  VERDICT: NO WIN — frozen features sufficient for this task")

print_comparison()


END-TO-END LoRA TRAINING (Stage 2)
  LoRA UNI v1 + SpatialMultRetNet: QWK = 0.9563 +/- 0.0027
  Fold QWKs: ['0.9531', '0.9530', '0.9585', '0.9592', '0.9578']

COMPARISON: LoRA vs Frozen Features
Method                                                 QWK
------------------------------------------------------------
  Frozen UNI v1 + ABMIL (V20)                 0.9460
  Frozen UNI v1 + RetNet (V20)                0.9540
  Frozen UNI v1 + SpatialMultRetNet (V20)     0.9520

  LoRA UNI v1 + SpatialMultRetNet (Stage 2)   0.9563 +/- 0.0027

  LoRA features + SpatialMultRetNet_bestHP    0.9498 +/- 0.0022
  LoRA features + RetNet_bestHP               0.9503 +/- 0.0041
  LoRA features + ABMIL_bestHP                0.9457 +/- 0.0059

────────────────────────────────────────────────────────────
  VERDICT: MARGINAL WIN — LoRA shows modest improvement
